<a href="https://colab.research.google.com/github/Jsolarte282000/app-recepcion/blob/main/BOT_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BLOQUE 0


In [ ]:
# ============================================================
# BLOQUE 0
# PREPARAR COLAB + SUBIR ARCHIVOS
# ============================================================

# Instalar librerías necesarias
%pip install -q xlrd openpyxl XlsxWriter

# Imports básicos
import pandas as pd
import re
import unicodedata

from pathlib import Path
from google.colab import files


# ============================================================
# SUBIR ARCHIVOS
# ============================================================

print("Selecciona los 3 archivos:")
print("1. BASE FDIM actualizada")
print("2. BASE VENTURE actualizada")
print("3. precios sister company actualizado")
print()

uploaded = files.upload()


# ============================================================
# CONTROL
# ============================================================

print("\n" + "=" * 70)
print("ARCHIVOS CARGADOS")
print("=" * 70)

for nombre in uploaded.keys():
    print("✅", nombre)

print("\nTotal archivos cargados:", len(uploaded))


if len(uploaded) < 3:
    print("\n⚠️ Faltan archivos. Deben cargarse los 3.")
else:
    print("\n✅ BLOQUE 0 OK")

Selecciona los 3 archivos:
1. BASE FDIM actualizada
2. BASE VENTURE actualizada
3. precios sister company actualizado



# BLOQUE 1

In [ ]:
# ============================================================
# BLOQUE 1
# CONFIGURACIÓN + REGLAS DE NEGOCIO
# ============================================================

from pathlib import Path


# ============================================================
# 1. IDENTIFICAR LOS ARCHIVOS SUBIDOS
# ============================================================

archivos = list(uploaded.keys())


def buscar_archivo(palabras):
    """
    Busca dentro de los archivos cargados uno cuyo nombre
    contenga todas las palabras indicadas.
    """
    for nombre in archivos:
        nombre_lower = nombre.lower()

        if all(p.lower() in nombre_lower for p in palabras):
            return Path("/content") / nombre

    return None


FDIM_PATH = buscar_archivo(["fdim"])
VENTURE_PATH = buscar_archivo(["venture"])
PRECIOS_PATH = buscar_archivo(["precios", "sister"])


# Archivo que generaremos al final
OUT_PATH = Path(
    "/content/VALIDACION_ASISTENTES_VENTAS.xlsx"
)


# ============================================================
# 2. VALIDAR QUE ENCONTRAMOS LOS 3 ARCHIVOS
# ============================================================

if FDIM_PATH is None:
    raise ValueError("❌ No encontré el archivo FDIM.")

if VENTURE_PATH is None:
    raise ValueError("❌ No encontré el archivo VENTURE.")

if PRECIOS_PATH is None:
    raise ValueError("❌ No encontré el archivo de precios Sister Company.")


# ============================================================
# 3. COLUMNAS IMPORTANTES DE VENTURE
# ============================================================

COL_ORDEN1 = "Orden\nCompra"
COL_ITEM1 = "No Item\n"
COL_DEST1 = "Destinatario\n"
COL_FECHA1 = "Fecha\nSalida"
COL_ETQ1 = "Codigo\nEtiqueta"
COL_CAJAS1 = "Cantidad\nCajas"

COL_PRECIO1 = "Precio\nUnitarios Sin Des"
COL_PRODUCTO1 = "Producto\n"
COL_VARIEDAD1 = "Variedad\n"


# ============================================================
# 4. REGLA WAREHOUSE FDIM -> CÓDIGO ETIQUETA
# ============================================================

MAPA_CODIGO_ETIQUETA = {

    "NO ASSIGNED": pd.NA,

    "EL MICKLETON NJ": "AMARILLA",
    "EL LEBANON, TN": "CAFE",
    "EL 120 S MIAMI": "FUCSIA",
    "EL BELFAIR, WA": "GRIS",
    "UB ITASCA, IL": "MORADA",
    "EL CITY OF INDUSTRY, CA": "NARANJA",
    "EL GARLAND, TX": "ROJA",
    "EL MIAMI, FL": "ROSADA",
    "EL 340 S MIAMI": "VERDE OBSCURO",

    # Nuevas reglas confirmadas
    "EL OTR S MIAMI": "SIN COLOR",
    "BS WILMINGTON, MA": "SIN COLOR",
}


# ============================================================
# 5. REGLA SISTER COMPANY
# ============================================================

# Sister Company aplica SOLAMENTE
# a derivaciones de TRADER JOE'S.

TOKEN_TRADER_JOES = "TRADERJOES"


# ============================================================
# 6. CONTROL
# ============================================================

print("=" * 75)
print("CONFIGURACIÓN")
print("=" * 75)

print("FDIM:")
print(" ", FDIM_PATH.name)

print("\nVENTURE:")
print(" ", VENTURE_PATH.name)

print("\nPRECIOS SISTER COMPANY:")
print(" ", PRECIOS_PATH.name)

print("\nArchivo de salida:")
print(" ", OUT_PATH.name)

print("\nWarehouses configurados:", len(MAPA_CODIGO_ETIQUETA))

print("\nRegla Sister Company:")
print(" ✅ Solo aplica a derivaciones TRADER JOE'S")

print("\n✅ BLOQUE 1 OK")

# BLOQUE 2

In [ ]:
# ============================================================
# BLOQUE 2
# FUNCIONES AUXILIARES
# ============================================================


# ============================================================
# 1. QUITAR TILDES
# ============================================================

def quitar_acentos(valor):

    if pd.isna(valor):
        return pd.NA

    valor = str(valor)

    return "".join(
        c
        for c in unicodedata.normalize("NFKD", valor)
        if not unicodedata.combining(c)
    )


# ============================================================
# 2. NORMALIZAR IDS
#
# Ejemplo:
# 24798998
# 24798998.0
# "24798998 "
#
# -> "24798998"
# ============================================================

def canon_id(valor):

    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip()

    if re.fullmatch(r"\d+\.0", valor):
        valor = valor[:-2]

    return valor


# ============================================================
# 3. ESTANDARIZAR DESTINATARIO
#
# Esta variable es INTERNA.
# No reemplaza el nombre original.
# ============================================================

def estandarizar_destinatario(valor):

    if pd.isna(valor):
        return pd.NA

    valor = str(valor).upper()

    # Quitar apóstrofes
    valor = re.sub(
        r"['’`´]",
        "",
        valor
    )

    # Quitar espacios
    valor = re.sub(
        r"\s+",
        "",
        valor
    )

    return valor


# ============================================================
# 4. TEXTO CANÓNICO
#
# Sirve para comparar:
#
# CAFE = CAFÉ
# "VERDE OBSCURO" = "VERDE-OBSCURO"
# ============================================================

def canon_texto(valor):

    if pd.isna(valor):
        return pd.NA

    valor = quitar_acentos(valor)

    valor = str(valor).upper().strip()

    # Solo letras y números
    valor = re.sub(
        r"[^A-Z0-9]",
        "",
        valor
    )

    return valor


# ============================================================
# 5. DESTINATARIO CANÓNICO
#
# Homologaciones ya confirmadas.
# ============================================================

def canon_destinatario(valor):

    valor = canon_texto(valor)

    if pd.isna(valor):
        return pd.NA

    # WHOLE FOODS MARKET -> WF
    valor = valor.replace(
        "WHOLEFOODSMARKET",
        "WF"
    )

    # WHOLE FOODS -> WF
    valor = valor.replace(
        "WHOLEFOODS",
        "WF"
    )

    # Otra variante posible
    valor = valor.replace(
        "WHOLEFOODMARKET",
        "WF"
    )

    return valor


# ============================================================
# 6. COMPARAR DESTINATARIOS
# ============================================================

def mismo_destinatario(a, b):

    a = canon_destinatario(a)
    b = canon_destinatario(b)

    # Ambos vacíos
    if pd.isna(a) and pd.isna(b):
        return True

    # Uno vacío y otro no
    if pd.isna(a) or pd.isna(b):
        return False

    # Coincidencia exacta
    if a == b:
        return True

    # --------------------------------------------------------
    # SUFIJOS OPERATIVOS YA CONFIRMADOS
    # --------------------------------------------------------

    sufijos_permitidos = [
        "MIAMI",
        "WM"
    ]

    for sufijo in sufijos_permitidos:

        if a == b + sufijo:
            return True

        if b == a + sufijo:
            return True

    return False


# ============================================================
# 7. DETECTAR TRADER JOE'S
#
# IMPORTANTE:
# Sister Company SOLO aplica aquí.
# ============================================================

def es_trader_joes(destinatario):

    valor = canon_destinatario(destinatario)

    if pd.isna(valor):
        return False

    return TOKEN_TRADER_JOES in valor


# ============================================================
# 8. COMPARACIÓN SEGURA
# ============================================================

def diferentes(a, b):

    if pd.isna(a) and pd.isna(b):
        return False

    if pd.isna(a) != pd.isna(b):
        return True

    return a != b


# ============================================================
# 9. CONSOLIDAR VALORES
#
# Si dentro de un Order + Item existe:
#
# SKU1
# SKU1
# SKU1
#
# -> SKU1
#
# Si existe:
#
# SKU1
# SKU2
#
# -> "SKU1 | SKU2"
#
# No se pierde información.
# ============================================================

def consolidar_valores(serie):

    valores = (
        serie
        .dropna()
        .astype(str)
        .str.strip()
    )

    valores = valores[
        valores != ""
    ]

    unicos = (
        valores
        .drop_duplicates()
        .tolist()
    )

    if len(unicos) == 0:
        return pd.NA

    if len(unicos) == 1:
        return unicos[0]

    return " | ".join(unicos)


# ============================================================
# 10. PRUEBAS RÁPIDAS
# ============================================================

print("=" * 75)
print("PRUEBAS BLOQUE 2")
print("=" * 75)

print(
    "CAFE vs CAFÉ:",
    canon_texto("CAFE") == canon_texto("CAFÉ")
)

print(
    "Whole Foods:",
    mismo_destinatario(
        "(SPD)(20005)SOUTHERNPACIFIC-VERNONWF",
        "(SPD)(20005)SOUTHERNPACIFIC-VERNON-WHOLEFOODS"
    )
)

print(
    "Trader Joe's:",
    es_trader_joes(
        "TRADER JOE'S LEBANON"
    )
)

print(
    "ID:",
    canon_id(24798998.0)
)

print("\n✅ BLOQUE 2 OK")

PRUEBAS BLOQUE 2
CAFE vs CAFÉ: True
Whole Foods: True
Trader Joe's: True
ID: 24798998

✅ BLOQUE 2 OK


# BLOQUE 3

In [ ]:
# ============================================================
# BLOQUE 3
# CARGAR + VALIDAR LAS 3 BASES MADRES
# ============================================================


# ============================================================
# 1. CARGAR ARCHIVOS
# ============================================================

print("=" * 80)
print("CARGANDO BASES...")
print("=" * 80)


df_venture_raw = pd.read_excel(
    VENTURE_PATH,
    sheet_name="Sheet1"
)

df_fdim_raw = pd.read_excel(
    FDIM_PATH,
    sheet_name="Orders"
)

df_precios_raw = pd.read_excel(
    PRECIOS_PATH,
    sheet_name="Hoja2"
)


print("✅ VENTURE cargado")
print("✅ FDIM cargado")
print("✅ PRECIOS SISTER COMPANY cargado")


# ============================================================
# 2. COLUMNAS OBLIGATORIAS VENTURE
# ============================================================

columnas_venture_requeridas = [

    COL_ORDEN1,
    COL_ITEM1,
    COL_DEST1,
    COL_FECHA1,
    COL_ETQ1,
    COL_CAJAS1,

    COL_PRECIO1,
    COL_PRODUCTO1,
    COL_VARIEDAD1
]


faltantes_venture = [
    col
    for col in columnas_venture_requeridas
    if col not in df_venture_raw.columns
]


# ============================================================
# 3. COLUMNAS OBLIGATORIAS FDIM
# ============================================================

columnas_fdim_requeridas = [

    "Order Number",
    "Item",
    "To",
    "Order Date",
    "Warehouse",
    "Quantity",

    "Status",
    "StatusDetail",
    "SKU",
    "Description (SKU)"
]


faltantes_fdim = [
    col
    for col in columnas_fdim_requeridas
    if col not in df_fdim_raw.columns
]


# ============================================================
# 4. COLUMNAS OBLIGATORIAS PRECIOS SISTER COMPANY
# ============================================================

columnas_precios_requeridas = [
    "ITEM",
    "TRADER"
]


faltantes_precios = [
    col
    for col in columnas_precios_requeridas
    if col not in df_precios_raw.columns
]


# ============================================================
# 5. DETENER SI CAMBIÓ LA ESTRUCTURA DE ALGUNA BASE
# ============================================================

errores = []


if faltantes_venture:

    errores.append(
        "VENTURE - faltan columnas:\n"
        + "\n".join(
            f"   • {repr(c)}"
            for c in faltantes_venture
        )
    )


if faltantes_fdim:

    errores.append(
        "FDIM - faltan columnas:\n"
        + "\n".join(
            f"   • {repr(c)}"
            for c in faltantes_fdim
        )
    )


if faltantes_precios:

    errores.append(
        "SISTER COMPANY - faltan columnas:\n"
        + "\n".join(
            f"   • {repr(c)}"
            for c in faltantes_precios
        )
    )


if errores:

    raise ValueError(
        "\n\n❌ CAMBIÓ LA ESTRUCTURA DE UNA BASE MADRE\n\n"
        + "\n\n".join(errores)
    )


# ============================================================
# 6. VALIDAR TABLA DE PRECIOS SISTER COMPANY
#
# Queremos evitar:
#
# GYP    0.29
# GYP    0.31
#
# sin que el sistema se dé cuenta.
# ============================================================

precios_control = df_precios_raw[
    ["ITEM", "TRADER"]
].copy()


# Normalizar ITEM únicamente para CONTROL
precios_control["_ITEM_CONTROL"] = (
    precios_control["ITEM"]
    .apply(canon_texto)
)


# Precio debe poder convertirse a número
precios_control["_PRECIO_CONTROL"] = pd.to_numeric(
    precios_control["TRADER"],
    errors="coerce"
)


# ------------------------------------------------------------
# PRECIOS NO NUMÉRICOS
# ------------------------------------------------------------

precios_no_numericos = precios_control[
    precios_control["_PRECIO_CONTROL"].isna()
    &
    precios_control["TRADER"].notna()
].copy()


# ------------------------------------------------------------
# ITEM VACÍO
# ------------------------------------------------------------

items_vacios = precios_control[
    precios_control["_ITEM_CONTROL"].isna()
].copy()


# ------------------------------------------------------------
# MISMO ITEM CON PRECIOS DISTINTOS
# ------------------------------------------------------------

control_duplicados = (
    precios_control
    .dropna(
        subset=[
            "_ITEM_CONTROL",
            "_PRECIO_CONTROL"
        ]
    )
    .groupby(
        "_ITEM_CONTROL"
    )["_PRECIO_CONTROL"]
    .nunique()
)


items_con_precios_distintos = (
    control_duplicados[
        control_duplicados > 1
    ]
    .index
    .tolist()
)


# ============================================================
# 7. SI HAY DUPLICADOS CON DISTINTO PRECIO -> DETENER
# ============================================================

if items_con_precios_distintos:

    detalle_conflictos = (
        precios_control[
            precios_control[
                "_ITEM_CONTROL"
            ].isin(
                items_con_precios_distintos
            )
        ][
            [
                "ITEM",
                "TRADER"
            ]
        ]
        .sort_values("ITEM")
    )

    print("\n❌ CONFLICTOS EN PRECIOS SISTER COMPANY")
    display(detalle_conflictos)

    raise ValueError(
        "Hay ITEM Sister Company con más de un precio diferente. "
        "Debemos corregir la base madre antes de continuar."
    )


# ============================================================
# 8. CONTROLES GENERALES
# ============================================================

print("\n" + "=" * 80)
print("RESUMEN DE BASES")
print("=" * 80)

print(
    f"VENTURE             : {len(df_venture_raw):,} filas"
)

print(
    f"FDIM                : {len(df_fdim_raw):,} filas"
)

print(
    f"SISTER COMPANY      : {len(df_precios_raw):,} filas"
)


print("\n" + "-" * 80)

print(
    "ITEM Sister con precios conflictivos:",
    len(items_con_precios_distintos)
)

print(
    "Precios no numéricos:",
    len(precios_no_numericos)
)

print(
    "ITEM vacíos en tabla de precios:",
    len(items_vacios)
)


# ============================================================
# 9. MOSTRAR TABLA DE PRECIOS CARGADA
# ============================================================

print("\n" + "=" * 80)
print("PRECIOS SISTER COMPANY CARGADOS")
print("=" * 80)

display(
    df_precios_raw[
        ["ITEM", "TRADER"]
    ].reset_index(drop=True)
)


# ============================================================
# 10. CONFIRMACIÓN FINAL
# ============================================================

print("\n✅ ESTRUCTURA VENTURE OK")
print("✅ ESTRUCTURA FDIM OK")
print("✅ ESTRUCTURA SISTER COMPANY OK")
print("✅ NO HAY ITEM CON PRECIOS CONFLICTIVOS")
print("\n✅ BLOQUE 3 OK")

CARGANDO BASES...
WARNING *** file size (592182) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
✅ VENTURE cargado
✅ FDIM cargado
✅ PRECIOS SISTER COMPANY cargado

RESUMEN DE BASES
VENTURE             : 2,335 filas
FDIM                : 1,069 filas
SISTER COMPANY      : 27 filas

--------------------------------------------------------------------------------
ITEM Sister con precios conflictivos: 0
Precios no numéricos: 0
ITEM vacíos en tabla de precios: 0

PRECIOS SISTER COMPANY CARGADOS


,ITEM,TRADER
0,ACHILEA,0.180
1,CRASPEDIA,0.185
2,DELPHINIUM,0.250
3,DISBUD : CREMON/SPIDER,0.250
4,ERYNGIUM,0.230
5,HYPERICUM,0.260
6,LEPIDIUM VIRGINICUM,0.210
7,LIMONIUM oshi y el skylight,0.220
8,LIMONIUM TINTED,0.245
9,LIMONIUM PIÑA COLADA,0.245



✅ ESTRUCTURA VENTURE OK
✅ ESTRUCTURA FDIM OK
✅ ESTRUCTURA SISTER COMPANY OK
✅ NO HAY ITEM CON PRECIOS CONFLICTIVOS

✅ BLOQUE 3 OK


# BLOQUE 4

In [ ]:
# ============================================================
# BLOQUE 4
# PREPARAR VENTURE CON DESPACHOS PARCIALES
#
# LLAVE OPERATIVA:
# Orden Compra + Item + Fecha Salida
#
# REGLAS CONFIRMADAS:
# - Fechas distintas = despachos distintos
# - Misma fecha = sumar cantidades
# - GYP = GYPSOPHILA
# - Sister Company solo aplica a familia TRADER_JOES
# ============================================================


# ============================================================
# 1. COPIA DE TRABAJO
# RAW QUEDA INTACTO
# ============================================================

df_venture_linea = df_venture_raw.copy()

# conservar orden original de las filas
df_venture_linea["_FILA_ORIGINAL"] = range(len(df_venture_linea))


# ============================================================
# 2. SEPARAR FILAS SIN ORDEN
# ============================================================

venture_sin_orden = df_venture_linea[
    df_venture_linea[COL_ORDEN1].isna()
].copy()

df_venture_linea = df_venture_linea[
    df_venture_linea[COL_ORDEN1].notna()
].copy()


# ============================================================
# 3. LLAVES ORDEN + ITEM
# ============================================================

df_venture_linea["_ORDEN_KEY"] = (
    df_venture_linea[COL_ORDEN1]
    .apply(canon_id)
)

df_venture_linea["_ITEM_KEY"] = (
    df_venture_linea[COL_ITEM1]
    .apply(canon_id)
)


# ============================================================
# 4. ITEM VACÍO -> ITEM 1
# Regla previamente validada
# ============================================================

mask_item_asumido = (
    df_venture_linea["_ORDEN_KEY"].notna()
    &
    df_venture_linea["_ITEM_KEY"].isna()
)

df_venture_linea.loc[
    mask_item_asumido,
    "_ITEM_KEY"
] = "1"

df_venture_linea["ITEM_ASUMIDO"] = mask_item_asumido


# ============================================================
# 5. DESTINATARIO TÉCNICO
# ============================================================

df_venture_linea["DESTINATARIO_STD"] = (
    df_venture_linea[COL_DEST1]
    .apply(estandarizar_destinatario)
)


# ============================================================
# 6. DATOS GENERALES DEL PEDIDO
#
# Destinatario y etiqueta pueden venir solo en una fila.
# Buscamos el primer valor real dentro de Orden + Item.
# ============================================================

def primer_valor_valido(serie):

    valores = serie.dropna()

    valores = valores[
        valores.astype(str).str.strip().ne("")
    ]

    if len(valores) == 0:
        return pd.NA

    return valores.iloc[0]


for columna in [
    COL_DEST1,
    "DESTINATARIO_STD",
    COL_ETQ1
]:

    df_venture_linea[
        columna + "_EFECTIVO"
    ] = (
        df_venture_linea
        .groupby(
            ["_ORDEN_KEY", "_ITEM_KEY"],
            dropna=False
        )[columna]
        .transform(primer_valor_valido)
    )


# ============================================================
# 7. FECHA EFECTIVA DEL DESPACHO
#
# MUY IMPORTANTE:
# La fecha se propaga hacia abajo solamente dentro
# de la misma Orden + Item.
#
# Así las variedades debajo de una línea pertenecen
# al despacho inmediatamente anterior.
# ============================================================

df_venture_linea["_FECHA_RAW"] = pd.to_datetime(
    df_venture_linea[COL_FECHA1],
    dayfirst=True,
    errors="coerce"
)

df_venture_linea["_FECHA_EFECTIVA"] = (
    df_venture_linea
    .groupby(
        ["_ORDEN_KEY", "_ITEM_KEY"],
        dropna=False
    )["_FECHA_RAW"]
    .ffill()
)


# ============================================================
# 8. LLAVE DE FECHA
# ============================================================

df_venture_linea["_FECHA_KEY"] = (
    df_venture_linea["_FECHA_EFECTIVA"]
    .dt.strftime("%Y-%m-%d")
)


# ============================================================
# 9. PRODUCTO ESTÁNDAR
#
# GYP = GYPSOPHILA
# ============================================================

def homologar_producto(valor):

    producto = canon_texto(valor)

    if pd.isna(producto):
        return pd.NA

    equivalencias = {
        "GYP": "GYPSOPHILA",
        "GYPSOPHILA": "GYPSOPHILA",
    }

    return equivalencias.get(
        producto,
        producto
    )


df_venture_linea["PRODUCTO_STD"] = (
    df_venture_linea[COL_PRODUCTO1]
    .apply(homologar_producto)
)


df_venture_linea["VARIEDAD_STD"] = (
    df_venture_linea[COL_VARIEDAD1]
    .apply(canon_texto)
)


# ============================================================
# 10. FAMILIA CLIENTE
# ============================================================

def identificar_familia_cliente(destinatario):

    if es_trader_joes(destinatario):
        return "TRADER_JOES"

    return "OTRO"


df_venture_linea["FAMILIA_CLIENTE"] = (
    df_venture_linea[
        "DESTINATARIO_STD_EFECTIVO"
    ]
    .apply(identificar_familia_cliente)
)


df_venture_linea["APLICA_SISTER_COMPANY"] = (
    df_venture_linea["FAMILIA_CLIENTE"]
    == "TRADER_JOES"
)


# ============================================================
# 11. CANTIDAD NUMÉRICA
#
# NO se propaga.
#
# Si existen:
#
# 11
# NaN
# NaN
# 8
# NaN
#
# para la misma Orden + Item + Fecha,
# el groupby posterior calcula 19.
# ============================================================

df_venture_linea["_CAJAS_NUM"] = pd.to_numeric(
    df_venture_linea[COL_CAJAS1],
    errors="coerce"
)


# ============================================================
# 12. CONTROL DE FILAS SIN FECHA RESUELTA
# ============================================================

filas_sin_fecha_resuelta = df_venture_linea[
    df_venture_linea["_FECHA_KEY"].isna()
].copy()


# ============================================================
# 13. DESPACHOS PARCIALES
#
# Cuántas fechas distintas tiene cada Orden + Item
# ============================================================

conteo_despachos = (
    df_venture_linea
    .dropna(subset=["_FECHA_KEY"])
    .groupby(
        ["_ORDEN_KEY", "_ITEM_KEY"]
    )["_FECHA_KEY"]
    .nunique()
    .rename("N_DESPACHOS_ORDEN_ITEM")
)


# ============================================================
# 14. CONSOLIDAR VENTURE
#
# NUEVA LLAVE:
#
# Orden + Item + Fecha
# ============================================================

grupos_despacho = df_venture_linea.dropna(
    subset=["_FECHA_KEY"]
).groupby(
    [
        "_ORDEN_KEY",
        "_ITEM_KEY",
        "_FECHA_KEY"
    ],
    dropna=False
)


df_venture_comp = grupos_despacho.agg(

    Orden_Compra=(
        COL_ORDEN1,
        "first"
    ),

    No_Item_Original=(
        COL_ITEM1,
        "first"
    ),

    Destinatario_Original=(
        COL_DEST1 + "_EFECTIVO",
        "first"
    ),

    DESTINATARIO_STD=(
        "DESTINATARIO_STD_EFECTIVO",
        "first"
    ),

    Codigo_Etiqueta=(
        COL_ETQ1 + "_EFECTIVO",
        "first"
    ),

    # REGLA NUEVA CONFIRMADA:
    # misma Orden + Item + Fecha -> SUM
    Cantidad_Cajas=(
        "_CAJAS_NUM",
        "sum"
    ),

    Productos=(
        COL_PRODUCTO1,
        consolidar_valores
    ),

    Variedades=(
        COL_VARIEDAD1,
        consolidar_valores
    ),

    N_LINEAS_PRODUCTO=(
        COL_PRODUCTO1,
        "size"
    ),

    N_REGISTROS_CANTIDAD=(
        "_CAJAS_NUM",
        "count"
    ),

    FAMILIA_CLIENTE=(
        "FAMILIA_CLIENTE",
        "first"
    ),

    APLICA_SISTER_COMPANY=(
        "APLICA_SISTER_COMPANY",
        "any"
    ),

    ITEM_ASUMIDO=(
        "ITEM_ASUMIDO",
        "any"
    )

).reset_index()


# ============================================================
# 15. AÑADIR NÚMERO DE DESPACHOS DEL PEDIDO
# ============================================================

df_venture_comp = df_venture_comp.merge(
    conteo_despachos.reset_index(),
    on=[
        "_ORDEN_KEY",
        "_ITEM_KEY"
    ],
    how="left"
)


df_venture_comp["ES_DESPACHO_PARCIAL"] = (
    df_venture_comp[
        "N_DESPACHOS_ORDEN_ITEM"
    ] > 1
)


df_venture_comp["CANTIDAD_SUMADA_MISMA_FECHA"] = (
    df_venture_comp[
        "N_REGISTROS_CANTIDAD"
    ] > 1
)


# ============================================================
# 16. CAMPOS PARA COMPARADOR
# ============================================================

df_venture_comp["_FECHA_B1"] = (
    df_venture_comp["_FECHA_KEY"]
)


df_venture_comp["_CAJAS_B1"] = pd.to_numeric(
    df_venture_comp["Cantidad_Cajas"],
    errors="coerce"
)


df_venture_comp["_ETIQUETA_B1"] = (
    df_venture_comp["Codigo_Etiqueta"]
    .apply(canon_texto)
)


# ============================================================
# 17. CONTROL CASOS QUE DESCUBRIMOS
# ============================================================

ordenes_control = [
    "24690118",
    "24761340",
    "24797935",
    "24797897"
]


print("=" * 85)
print("CONTROL DE LOS 4 CASOS QUE DEFINIERON LA NUEVA REGLA")
print("=" * 85)


display(
    df_venture_comp.loc[
        df_venture_comp[
            "_ORDEN_KEY"
        ].isin(ordenes_control),

        [
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY",
            "Cantidad_Cajas",
            "N_REGISTROS_CANTIDAD",
            "N_DESPACHOS_ORDEN_ITEM",
            "ES_DESPACHO_PARCIAL",
            "CANTIDAD_SUMADA_MISMA_FECHA"
        ]
    ]
    .sort_values(
        [
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 18. CONTROL TRADER JOE'S
# ============================================================

trader_joes_detectados = (
    df_venture_linea.loc[
        df_venture_linea["APLICA_SISTER_COMPANY"],
        [
            COL_DEST1 + "_EFECTIVO",
            "FAMILIA_CLIENTE"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


print("\nDERIVACIONES TRADER JOE'S:")
display(trader_joes_detectados)


# ============================================================
# 19. RESUMEN
# ============================================================

print("\n" + "=" * 85)
print("RESUMEN BLOQUE 4 NUEVO")
print("=" * 85)

print(
    "Filas RAW VENTURE:",
    len(df_venture_raw)
)

print(
    "Filas producto utilizables:",
    len(df_venture_linea)
)

print(
    "Despachos únicos Orden + Item + Fecha:",
    len(df_venture_comp)
)

print(
    "Despachos parciales:",
    int(
        df_venture_comp[
            "ES_DESPACHO_PARCIAL"
        ].sum()
    )
)

print(
    "Despachos donde se sumaron varias cantidades en la misma fecha:",
    int(
        df_venture_comp[
            "CANTIDAD_SUMADA_MISMA_FECHA"
        ].sum()
    )
)

print(
    "Filas sin fecha resuelta:",
    len(filas_sin_fecha_resuelta)
)

print(
    "Filas sin Orden Compra:",
    len(venture_sin_orden)
)

print(
    "Líneas Trader Joe's:",
    int(
        df_venture_linea[
            "APLICA_SISTER_COMPANY"
        ].sum()
    )
)


print("\n✅ BLOQUE 4 NUEVO OK")

CONTROL DE LOS 4 CASOS QUE DEFINIERON LA NUEVA REGLA


,_ORDEN_KEY,_ITEM_KEY,_FECHA_KEY,Cantidad_Cajas,N_REGISTROS_CANTIDAD,N_DESPACHOS_ORDEN_ITEM,ES_DESPACHO_PARCIAL,CANTIDAD_SUMADA_MISMA_FECHA
0,24690118,2,2026-08-20,4.0,1,2,True,False
1,24690118,2,2026-08-21,9.0,1,2,True,False
2,24761340,6,2026-08-20,10.0,1,2,True,False
3,24761340,6,2026-08-21,15.0,1,2,True,False
4,24797897,4,2026-08-20,19.0,2,1,False,True
5,24797935,1,2026-08-20,2.0,1,2,True,False
6,24797935,1,2026-08-21,6.0,1,2,True,False



DERIVACIONES TRADER JOE'S:


,Destinatario\n_EFECTIVO,FAMILIA_CLIENTE
0,TRADER JOE´S WASHINGTON,TRADER_JOES
1,TRADER JOE'S CALIFORNIA,TRADER_JOES
2,TRADER JOE'S CHICAGO,TRADER_JOES
3,TRADER JOE'S LEBANON,TRADER_JOES
4,TRADER JOE'S MIAMI,TRADER_JOES
5,TRADER JOES NEW JERSEY,TRADER_JOES
6,TRADER JOE'S TEXAS,TRADER_JOES



RESUMEN BLOQUE 4 NUEVO
Filas RAW VENTURE: 2335
Filas producto utilizables: 2256
Despachos únicos Orden + Item + Fecha: 366
Despachos parciales: 6
Despachos donde se sumaron varias cantidades en la misma fecha: 4
Filas sin fecha resuelta: 0
Filas sin Orden Compra: 79
Líneas Trader Joe's: 656

✅ BLOQUE 4 NUEVO OK
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
# ============================================================
# BLOQUE 5
# PREPARAR FDIM
#
# OBJETIVOS:
# 1. Mantener df_fdim_raw intacto
# 2. Crear Orden + Item + Fecha
# 3. Convertir Warehouse -> Codigo Etiqueta
# 4. Preparar Quantity numérico
# 5. Revisar estructura antes de consolidar
# ============================================================


# ============================================================
# 1. COPIA DE TRABAJO
# ============================================================

df_fdim_linea = df_fdim_raw.copy()

df_fdim_linea["_FILA_ORIGINAL"] = range(
    len(df_fdim_linea)
)


print("=" * 85)
print("BLOQUE 5 - PREPARANDO FDIM")
print("=" * 85)

print(
    "Filas RAW FDIM:",
    len(df_fdim_raw)
)


# ============================================================
# 2. LLAVES ORDEN + ITEM
# ============================================================

df_fdim_linea["_ORDEN_KEY"] = (
    df_fdim_linea["Order Number"]
    .apply(canon_id)
)


df_fdim_linea["_ITEM_KEY"] = (
    df_fdim_linea["Item"]
    .apply(canon_id)
)


# ============================================================
# 3. FECHA FDIM
#
# IMPORTANTE:
# FDIM viene DD/MM/YYYY.
#
# 21/08/2026
# ->
# 2026-08-21
# ============================================================

df_fdim_linea["_FECHA_RAW"] = pd.to_datetime(
    df_fdim_linea["Order Date"],
    dayfirst=True,
    errors="coerce"
)


df_fdim_linea["_FECHA_KEY"] = (
    df_fdim_linea["_FECHA_RAW"]
    .dt.strftime("%Y-%m-%d")
)


# ============================================================
# 4. DESTINATARIO
# ============================================================

df_fdim_linea["DESTINATARIO_STD"] = (
    df_fdim_linea["To"]
    .apply(estandarizar_destinatario)
)


# ============================================================
# 5. WAREHOUSE ESTÁNDAR
# ============================================================

df_fdim_linea["WAREHOUSE_STD"] = (
    df_fdim_linea["Warehouse"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# ============================================================
# 6. WAREHOUSE -> CÓDIGO ETIQUETA
# ============================================================

df_fdim_linea["Codigo_Etiqueta_B2"] = (
    df_fdim_linea["WAREHOUSE_STD"]
    .map(MAPA_CODIGO_ETIQUETA)
)


df_fdim_linea["_ETIQUETA_B2"] = (
    df_fdim_linea["Codigo_Etiqueta_B2"]
    .apply(canon_texto)
)


# ============================================================
# 7. QUANTITY NUMÉRICO
# ============================================================

df_fdim_linea["_QTY_NUM"] = pd.to_numeric(
    df_fdim_linea["Quantity"],
    errors="coerce"
)


# ============================================================
# 8. WAREHOUSES SIN MAPEO
#
# NO ASSIGNED no cuenta como error.
# ============================================================

warehouses_sin_mapeo = (
    df_fdim_linea.loc[
        df_fdim_linea[
            "Codigo_Etiqueta_B2"
        ].isna()
        &
        df_fdim_linea[
            "WAREHOUSE_STD"
        ].ne("NO ASSIGNED")
        &
        df_fdim_linea[
            "WAREHOUSE_STD"
        ].notna(),
        "WAREHOUSE_STD"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


# ============================================================
# 9. CUÁNTAS FECHAS TIENE CADA ORDER + ITEM
# ============================================================

fechas_por_order_item_fdim = (
    df_fdim_linea
    .dropna(
        subset=[
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY"
        ]
    )
    .groupby(
        [
            "_ORDEN_KEY",
            "_ITEM_KEY"
        ]
    )["_FECHA_KEY"]
    .nunique()
    .rename("N_FECHAS_FDIM")
)


orden_item_multi_fecha_fdim = (
    fechas_por_order_item_fdim[
        fechas_por_order_item_fdim > 1
    ]
)


# ============================================================
# 10. CONTROL DE QUANTITY POSITIVO
#
# Queremos verificar la estructura técnica FDIM
# ahora usando:
#
# Orden + Item + Fecha
# ============================================================

control_qty_fdim = (
    df_fdim_linea
    .dropna(
        subset=[
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY"
        ]
    )
    .groupby(
        [
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY"
        ]
    )["_QTY_NUM"]
    .agg(
        FILAS="size",
        QTY_POSITIVOS=lambda s: (s > 0).sum(),
        SUM_QUANTITY="sum",
        MAX_QUANTITY="max"
    )
    .reset_index()
)


grupos_con_mas_de_un_qty_positivo = (
    control_qty_fdim[
        control_qty_fdim[
            "QTY_POSITIVOS"
        ] > 1
    ]
    .copy()
)


# ============================================================
# 11. CONTROL DE LAS 4 ÓRDENES CLAVE
# ============================================================

ordenes_control = [
    "24690118",
    "24761340",
    "24797935",
    "24797897"
]


print("\n" + "=" * 85)
print("LAS 4 ÓRDENES EN FDIM")
print("=" * 85)


control_4_ordenes_fdim = (
    df_fdim_linea.loc[
        df_fdim_linea[
            "_ORDEN_KEY"
        ].isin(ordenes_control),

        [
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY",

            "To",
            "Warehouse",
            "Codigo_Etiqueta_B2",

            "Quantity",
            "_QTY_NUM",

            "Status",
            "StatusDetail"
        ]
    ]

    .sort_values(
        [
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY"
        ]
    )

    .reset_index(drop=True)
)


display(
    control_4_ordenes_fdim
)


# ============================================================
# 12. RESUMEN
# ============================================================

print("\n" + "=" * 85)
print("RESUMEN BLOQUE 5")
print("=" * 85)


print(
    "Filas RAW FDIM:",
    len(df_fdim_raw)
)


print(
    "Order + Item con más de una fecha:",
    len(orden_item_multi_fecha_fdim)
)


print(
    "Order + Item + Fecha únicos:",
    len(control_qty_fdim)
)


print(
    "Grupos con más de un Quantity positivo:",
    len(grupos_con_mas_de_un_qty_positivo)
)


print(
    "Warehouses sin mapeo:",
    warehouses_sin_mapeo
)


print(
    "Filas con fecha no interpretable:",
    int(
        df_fdim_linea[
            "_FECHA_KEY"
        ].isna().sum()
    )
)


print("\n✅ BLOQUE 5 OK")

BLOQUE 5 - PREPARANDO FDIM
Filas RAW FDIM: 1069

LAS 4 ÓRDENES EN FDIM


,_ORDEN_KEY,_ITEM_KEY,_FECHA_KEY,To,Warehouse,Codigo_Etiqueta_B2,Quantity,_QTY_NUM,Status,StatusDetail



RESUMEN BLOQUE 5
Filas RAW FDIM: 1069
Order + Item con más de una fecha: 0
Order + Item + Fecha únicos: 338
Grupos con más de un Quantity positivo: 0
Warehouses sin mapeo: []
Filas con fecha no interpretable: 0

✅ BLOQUE 5 OK


In [ ]:
# ============================================================
# CONTROL 5A
# BUSCAR LAS 4 ORDENES EN FDIM
# NO MODIFICA DATOS
# ============================================================

ordenes_buscar = [
    "24690118",
    "24761340",
    "24797935",
    "24797897"
]

# Crear versión comparable del Order Number
df_control_5a = df_fdim_linea.copy()

df_control_5a["_ORDEN_CONTROL"] = (
    df_control_5a["Order Number"]
    .apply(canon_id)
)

resultado_busqueda = df_control_5a[
    df_control_5a["_ORDEN_CONTROL"].isin(ordenes_buscar)
][
    [
        "Order Number",
        "Item",
        "Order Date",
        "Quantity",
        "To",
        "Warehouse",
        "Status",
        "StatusDetail"
    ]
].copy()


print("=" * 80)
print("CONTROL 5A - BUSQUEDA DE ORDENES EN FDIM")
print("=" * 80)

print("Filas encontradas:", len(resultado_busqueda))

display(resultado_busqueda)

CONTROL 5A - BUSQUEDA DE ORDENES EN FDIM
Filas encontradas: 0


,Order Number,Item,Order Date,Quantity,To,Warehouse,Status,StatusDetail


# BLOQUE 6

In [ ]:
# ============================================================
# BLOQUE 6
# CONSOLIDAR FDIM
#
# LLAVE:
# Order Number + Item + Order Date
#
# Quantity = SUM
# ============================================================


print("=" * 85)
print("BLOQUE 6 - CONSOLIDANDO FDIM")
print("=" * 85)


# ============================================================
# 1. CAMPOS FDIM QUE QUEREMOS CONSERVAR
# ============================================================

campos_info_fdim = [

    "Order Number",
    "Item",

    "To",
    "DESTINATARIO_STD",

    "Order Date",

    "Warehouse",
    "WAREHOUSE_STD",
    "Codigo_Etiqueta_B2",

    "Status",
    "StatusDetail",

    "Ship Date",
    "QuantityConfirmed",

    "Suppliers",

    "SKU",
    "Description (SKU)",

    "Unit",

    "Long",
    "Height",
    "Width",
    "Weight",

    "Directions",

    "Third Code",
    "UPC Code",

    "BoxBrand",
    "Box Type",
    "Category",

    "Barcode Number",
    "Barcode Date",

    "Tipo unidad",

    "CantidadTallosPorRamo",
    "TotalTallosPorCaja",
    "TotalTallosPorItem",
    "CantidadRamosPorCaja",
    "Total ramos"
]


# Solo conservar columnas que realmente existan
campos_info_fdim = [
    c
    for c in campos_info_fdim
    if c in df_fdim_linea.columns
]


# ============================================================
# 2. GRUPOS
# ============================================================

grupos_fdim = (

    df_fdim_linea
    .dropna(
        subset=[
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY"
        ]
    )
    .groupby(
        [
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY"
        ],
        dropna=False
    )
)


# ============================================================
# 3. CONSOLIDAR INFORMACIÓN
#
# Si hay un solo valor -> conservar
#
# Si aparecen varios:
# A | B
#
# No perdemos información.
# ============================================================

datos_fdim = (

    grupos_fdim[campos_info_fdim]

    .agg(
        {
            c: consolidar_valores
            for c in campos_info_fdim
        }
    )

    .reset_index()
)


# ============================================================
# 4. CONSOLIDAR QUANTITY
# ============================================================

cantidades_fdim = (

    grupos_fdim["_QTY_NUM"]

    .agg(

        Quantity=lambda s:
            s.sum(min_count=1),

        FILAS_TECNICAS_FDIM="size",

        QTY_POSITIVOS_FDIM=lambda s:
            int((s > 0).sum())

    )

    .reset_index()
)


# ============================================================
# 5. UNIR
# ============================================================

df_fdim_comp = datos_fdim.merge(

    cantidades_fdim,

    on=[
        "_ORDEN_KEY",
        "_ITEM_KEY",
        "_FECHA_KEY"
    ],

    how="left"
)


# ============================================================
# 6. CAMPOS PARA EL COMPARADOR
# ============================================================

df_fdim_comp["_FECHA_B2"] = (
    df_fdim_comp["_FECHA_KEY"]
)


df_fdim_comp["_CAJAS_B2"] = pd.to_numeric(
    df_fdim_comp["Quantity"],
    errors="coerce"
)


df_fdim_comp["_ETIQUETA_B2"] = (
    df_fdim_comp["Codigo_Etiqueta_B2"]
    .apply(canon_texto)
)


# ============================================================
# 7. CONTROL DE DUPLICADOS
# ============================================================

duplicados_finales_fdim = (

    df_fdim_comp

    .duplicated(
        subset=[
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY"
        ],
        keep=False
    )

    .sum()
)


# ============================================================
# 8. RESUMEN
# ============================================================

print("\nFilas FDIM originales:", len(df_fdim_linea))

print(
    "Despachos únicos Orden + Item + Fecha:",
    len(df_fdim_comp)
)

print(
    "Duplicados después de consolidar:",
    int(duplicados_finales_fdim)
)

print(
    "Grupos con varias filas técnicas:",
    int(
        (
            df_fdim_comp[
                "FILAS_TECNICAS_FDIM"
            ] > 1
        ).sum()
    )
)

print(
    "Grupos con más de un Quantity positivo:",
    int(
        (
            df_fdim_comp[
                "QTY_POSITIVOS_FDIM"
            ] > 1
        ).sum()
    )
)


# ============================================================
# 9. MUESTRA
# ============================================================

print("\nEJEMPLO FDIM CONSOLIDADO:")

display(

    df_fdim_comp[
        [
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY",
            "To",
            "Warehouse",
            "Codigo_Etiqueta_B2",
            "Quantity",
            "FILAS_TECNICAS_FDIM",
            "QTY_POSITIVOS_FDIM"
        ]
    ]
    .head(20)
)


print("\n✅ BLOQUE 6 OK")

BLOQUE 6 - CONSOLIDANDO FDIM

Filas FDIM originales: 1069
Despachos únicos Orden + Item + Fecha: 338
Duplicados después de consolidar: 0
Grupos con varias filas técnicas: 124
Grupos con más de un Quantity positivo: 0

EJEMPLO FDIM CONSOLIDADO:


,_ORDEN_KEY,_ITEM_KEY,_FECHA_KEY,To,Warehouse,Codigo_Etiqueta_B2,Quantity,FILAS_TECNICAS_FDIM,QTY_POSITIVOS_FDIM
0,24262110,26,2026-08-20,EVERFLORA,"EL Miami, FL",ROSADA,1,1,1
1,24262110,29,2026-08-20,EVERFLORA,"EL Miami, FL",ROSADA,1,1,1
2,24262146,2,2026-08-20,MIAMI OFFICE ACCOUNT,"EL Miami, FL",ROSADA,1,1,1
3,24262146,4,2026-08-20,MIAMI OFFICE ACCOUNT,"EL Miami, FL",ROSADA,5,1,1
4,24262146,5,2026-08-20,MIAMI OFFICE ACCOUNT,"EL Miami, FL",ROSADA,3,1,1
5,24262158,1,2026-08-20,EVERFLORA,"EL Miami, FL",ROSADA,1,5,1
6,24262158,2,2026-08-20,EVERFLORA,"EL Miami, FL",ROSADA,1,5,1
7,24262183,23,2026-08-21,ASHLAND ADDISON,"EL Miami, FL",ROSADA,1,1,1
8,24262183,31,2026-08-21,ASHLAND ADDISON,"EL Miami, FL",ROSADA,1,1,1
9,24262220,1,2026-08-21,ORIGINAL HEROMAN´S FLORIST,"EL Miami, FL",ROSADA,1,1,1



✅ BLOQUE 6 OK


# BLOQUE 7

In [ ]:
# ============================================================
# AJUSTE 7C
# HOMOLOGACIÓN ROBUSTA WHOLE FOODS
# ============================================================

def es_whole_foods(valor):

    if pd.isna(valor):
        return False

    texto = quitar_acentos(str(valor)).upper()

    return (
        "WHOLE FOODS" in texto
        or "WHOLEFOODS" in texto
        or "WHOLE FOOD" in texto
        or re.search(r"\bWF\b", texto) is not None
    )


def codigo_cliente(valor):

    if pd.isna(valor):
        return None

    texto = str(valor)

    # Busca códigos como (20013), (20006), (20017)
    numeros = re.findall(r"\((\d{4,6})\)", texto)

    if numeros:
        return numeros[-1]

    return None


def mismo_destinatario_v3(a, b):

    # Toda la lógica anterior
    if mismo_destinatario_v2(a, b):
        return True

    # --------------------------------------------------------
    # WHOLE FOODS
    # --------------------------------------------------------

    if es_whole_foods(a) and es_whole_foods(b):

        codigo_a = codigo_cliente(a)
        codigo_b = codigo_cliente(b)

        # Si ambos tienen el mismo código comercial
        if (
            codigo_a is not None
            and codigo_b is not None
            and codigo_a == codigo_b
        ):
            return True

        # Segunda comparación:
        # WHOLE FOODS / WHOLE FOODS MARKET / WF = WF

        a_std = canon_destinatario(a)
        b_std = canon_destinatario(b)

        # Evitar WF duplicado: WFWF -> WF
        while "WFWF" in a_std:
            a_std = a_std.replace("WFWF", "WF")

        while "WFWF" in b_std:
            b_std = b_std.replace("WFWF", "WF")

        if a_std == b_std:
            return True

    return False


print("✅ AJUSTE WHOLE FOODS CARGADO")

✅ AJUSTE WHOLE FOODS CARGADO


In [ ]:
# ============================================================
# AJUSTE 7D
# NORMALIZACIÓN DE NOMBRES OPERATIVOS
# ============================================================

def nombre_operativo(valor):

    if pd.isna(valor):
        return pd.NA

    x = quitar_acentos(str(valor)).upper()
    # Quitar puntuación / espacios
    x = re.sub(r"[^A-Z0-9]", "", x)

    # Equivalencias comerciales
    x = x.replace("WHOLE FOODS MARKET", "WF")
    x = x.replace("WHOLE FOOD MARKET", "WF")
    x = x.replace("WHOLE FOODS", "WF")
    x = x.replace("FALCOM FARMS", "FALCON FARMS")

    x = x.replace("STOP Y SHOP", "STOP SHOP")

    # Descripciones operativas equivalentes
    x = x.replace("DISTRIBUTION CENTER", "DISTRIBUTION")
    x = x.replace("SUPERMARKETS", "SUPERMARKET")

    # Quitar puntuación / espacios
    x = re.sub(r"[^A-Z0-9]", "", x)

    # Evitar WF duplicado
    while "WFWF" in x:
        x = x.replace("WFWF", "WF")

    # Sufijos operativos que no cambian el cliente
    for sufijo in ["WF", "WM", "MIA"]:
        if x.endswith(sufijo) and len(x) > len(sufijo) + 8:
            x = x[:-len(sufijo)]

    return x


def mismo_destinatario_v4(a, b):

    # Mantener todas las reglas anteriores
    if mismo_destinatario_v3(a, b):
        return True

    a2 = nombre_operativo(a)
    b2 = nombre_operativo(b)

    if pd.isna(a2) or pd.isna(b2):
        return False

    return a2 == b2


print("✅ AJUSTE 7D CARGADO")

✅ AJUSTE 7D CARGADO


In [ ]:
# ============================================================
# AJUSTE 7F
# NORMALIZACION OPERATIVA FINAL DE DESTINATARIOS
# ============================================================

def nombre_operativo_v2(valor):

    if pd.isna(valor):
        return pd.NA

    x = quitar_acentos(str(valor)).upper().strip()


    # --------------------------------------------------------
    # ERRORES / VARIANTES CONFIRMADAS
    # --------------------------------------------------------

    # FALCOM = FALCON
    x = x.replace(
        "FALCOM",
        "FALCON"
    )


    # --------------------------------------------------------
    # STOP & SHOP
    #
    # STOP Y SHOP
    # STOP - SHOP
    # STOP SHOP
    #
    # -> STOPSHOP
    # --------------------------------------------------------

    x = re.sub(
        r"\bSTOP\s*(?:Y|-)?\s*SHOP\b",
        "STOPSHOP",
        x
    )


    # --------------------------------------------------------
    # WHOLE FOODS
    # --------------------------------------------------------

    x = x.replace(
        "WHOLE FOODS MARKET",
        "WF"
    )

    x = x.replace(
        "WHOLE FOOD MARKET",
        "WF"
    )

    x = x.replace(
        "WHOLE FOODS",
        "WF"
    )


    # Caso:
    # WHOLE WF
    # -> WF
    x = re.sub(
        r"\bWHOLE\s*[-/]?\s*WF\b",
        "WF",
        x
    )


    # --------------------------------------------------------
    # DISTRIBUTION CENTER
    # =
    # DISTRIBUTION
    # --------------------------------------------------------

    x = x.replace(
        "DISTRIBUTION CENTER",
        "DISTRIBUTION"
    )


    # --------------------------------------------------------
    # SUPERMARKETS = SUPERMARKET
    # --------------------------------------------------------

    x = x.replace(
        "SUPERMARKETS",
        "SUPERMARKET"
    )


    # --------------------------------------------------------
    # QUITAR PUNTUACION Y ESPACIOS
    # --------------------------------------------------------

    x = re.sub(
        r"[^A-Z0-9]",
        "",
        x
    )


    # --------------------------------------------------------
    # EVITAR WF REPETIDO
    # --------------------------------------------------------

    while "WFWF" in x:
        x = x.replace(
            "WFWF",
            "WF"
        )


    # --------------------------------------------------------
    # SUFIJOS OPERATIVOS
    # --------------------------------------------------------

    for sufijo in [
        "WF",
        "WM",
        "MIA"
    ]:

        if (
            x.endswith(sufijo)
            and
            len(x) > len(sufijo) + 8
        ):

            x = x[:-len(sufijo)]


    return x


# ============================================================
# COMPARADOR V5
# ============================================================

def mismo_destinatario_v5(a, b):

    # Conservar todas las reglas anteriores
    if mismo_destinatario_v4(a, b):
        return True

    a2 = nombre_operativo_v2(a)
    b2 = nombre_operativo_v2(b)

    if pd.isna(a2) or pd.isna(b2):
        return False

    return a2 == b2


print("✅ AJUSTE 7F CARGADO")

✅ AJUSTE 7F CARGADO


In [ ]:
# ============================================================
# BLOQUE 7B
# AJUSTES DE NEGOCIO
#
# 1. ADUSA + mismo FC = mismo destinatario
# 2. FDIM sin color NO genera cambio
#    Observación = SIN COLOR FDIM
# ============================================================


# ============================================================
# 1. HOMOLOGACIÓN ADUSA
# ============================================================

def mismo_destinatario_v2(a, b):

    # Primero usar toda nuestra lógica anterior
    if mismo_destinatario(a, b):
        return True

    a_std = canon_destinatario(a)
    b_std = canon_destinatario(b)

    if pd.isna(a_std) or pd.isna(b_std):
        return False

    # --------------------------------------------------------
    # ADUSA
    #
    # Ejemplo:
    # ADUSA DISTRIBUTION LLC - FC#54
    # ADUSA DISTRIBUTION, LLC - FC#54
    #
    # Si ambos son ADUSA y tienen el mismo FC,
    # son el mismo cliente.
    # --------------------------------------------------------

    if "ADUSA" in a_std and "ADUSA" in b_std:

        fc_a = re.search(r"FC(\d+)", a_std)
        fc_b = re.search(r"FC(\d+)", b_std)

        if fc_a and fc_b:

            return (
                fc_a.group(1)
                ==
                fc_b.group(1)
            )

    return False


# ============================================================
# 2. DETECTAR FDIM SIN COLOR
# ============================================================

def fdim_sin_color(valor):

    if pd.isna(valor):
        return True

    valor_std = canon_texto(valor)

    if pd.isna(valor_std):
        return True

    return valor_std in [
        "",
        "SINCOLOR"
    ]


# ============================================================
# 3. RECALCULAR FILA
# ============================================================

def recalcular_7b(row):

    # --------------------------------------------------------
    # NO ENCONTRADO
    # --------------------------------------------------------

    if not row["EXISTE_EN_FDIM"]:

        return pd.Series({

            "RESULTADO":
                "NO ENCONTRADO",

            "CAMPOS_CAMBIADOS":
                "",

            "NUM_CAMBIOS":
                0,

            "DETALLE_CAMBIOS":
                "Orden + Item no existe en FDIM",

            "OBSERVACION_FDIM":
                ""

        })


    cambios = []
    detalles = []
    observaciones = []


    # ========================================================
    # FECHA
    # ========================================================

    if diferentes(
        row["_FECHA_B1"],
        row["FECHA_FDIM"]
    ):

        cambios.append("FECHA")

        detalles.append(
            f"Fecha: "
            f"{row['_FECHA_B1']} -> "
            f"{row['FECHA_FDIM']}"
        )


    # ========================================================
    # DESTINATARIO
    # ========================================================

    if not mismo_destinatario_v5(
    row["Destinatario_Original"],
    row["Destinatario_FDIM"]
    ):

        cambios.append(
            "DESTINATARIO"
        )

        detalles.append(
            f"Destinatario: "
            f"{row['Destinatario_Original']} -> "
            f"{row['Destinatario_FDIM']}"
        )


    # ========================================================
    # ETIQUETA
    #
    # REGLA NUEVA:
    #
    # FDIM vacío / SIN COLOR
    # NO genera CAMBIO.
    # ========================================================

    if fdim_sin_color(
        row["Codigo_Etiqueta_FDIM"]
    ):

        observaciones.append(
            "SIN COLOR FDIM"
        )

    else:

        if diferentes(
            row["_ETIQUETA_B1"],
            row["_ETIQUETA_B2"]
        ):

            cambios.append(
                "CODIGO ETIQUETA"
            )

            detalles.append(
                f"Etiqueta: "
                f"{row['Codigo_Etiqueta']} -> "
                f"{row['Codigo_Etiqueta_FDIM']}"
            )


    # ========================================================
    # CAJAS
    # ========================================================

    if diferentes(
        row["_CAJAS_B1"],
        row["_CAJAS_B2"]
    ):

        cambios.append(
            "CANT. CAJAS"
        )

        if (
            pd.notna(row["_CAJAS_B1"])
            and
            pd.notna(row["_CAJAS_B2"])
        ):

            delta = (
                row["_CAJAS_B2"]
                -
                row["_CAJAS_B1"]
            )

            detalles.append(
                f"Cajas: "
                f"{row['_CAJAS_B1']:g} -> "
                f"{row['_CAJAS_B2']:g} "
                f"(Delta {delta:+g})"
            )


    # ========================================================
    # RESULTADO
    # ========================================================

    resultado = (
        "SIN CAMBIO"
        if len(cambios) == 0
        else "CAMBIO"
    )

    detalle = (
        "Sin cambios"
        if len(detalles) == 0
        else " || ".join(detalles)
    )


    return pd.Series({

        "RESULTADO":
            resultado,

        "CAMPOS_CAMBIADOS":
            " | ".join(cambios),

        "NUM_CAMBIOS":
            len(cambios),

        "DETALLE_CAMBIOS":
            detalle,

        "OBSERVACION_FDIM":
            " | ".join(observaciones)

    })


# ============================================================
# 4. BORRAR RESULTADO VIEJO
# ============================================================

columnas_recalcular = [

    "RESULTADO",
    "CAMPOS_CAMBIADOS",
    "NUM_CAMBIOS",
    "DETALLE_CAMBIOS",
    "OBSERVACION_FDIM"

]


df_comparacion = df_comparacion.drop(
    columns=[
        c
        for c in columnas_recalcular
        if c in df_comparacion.columns
    ]
)


# ============================================================
# 5. RECALCULAR
# ============================================================

resultado_7b = (
    df_comparacion
    .apply(
        recalcular_7b,
        axis=1
    )
)


df_comparacion = pd.concat(
    [
        df_comparacion,
        resultado_7b
    ],
    axis=1
)


# ============================================================
# 6. RESULTADO
# ============================================================

print("=" * 85)
print("RESULTADO DESPUÉS DE AJUSTES 7B")
print("=" * 85)


display(
    df_comparacion[
        "RESULTADO"
    ]
    .value_counts()
    .rename_axis("RESULTADO")
    .reset_index(name="CANTIDAD")
)


print("\nCasos SIN COLOR FDIM:")

print(
    (
        df_comparacion[
            "OBSERVACION_FDIM"
        ]
        ==
        "SIN COLOR FDIM"
    ).sum()
)


print("\n✅ BLOQUE 7B OK")

RESULTADO DESPUÉS DE AJUSTES 7B


,RESULTADO,CANTIDAD
0,SIN CAMBIO,265
1,NO ENCONTRADO,61
2,CAMBIO,40



Casos SIN COLOR FDIM:
6

✅ BLOQUE 7B OK


In [ ]:
# ============================================================
# CONTROL 7A
# REVISAR CAMBIOS DE DESTINATARIO / ETIQUETA
# NO MODIFICA DATOS
# ============================================================

mask_control_7a = (
    (df_comparacion["RESULTADO"] == "CAMBIO")
    &
    (
        df_comparacion["CAMPOS_CAMBIADOS"].str.contains(
            "DESTINATARIO",
            na=False
        )
        |
        df_comparacion["CAMPOS_CAMBIADOS"].str.contains(
            "CODIGO ETIQUETA",
            na=False
        )
    )
)


df_control_7a = df_comparacion.loc[
    mask_control_7a,
    [
        "_ORDEN_KEY",
        "_ITEM_KEY",

        "_FECHA_B1",
        "FECHA_FDIM",

        "Destinatario_Original",
        "Destinatario_FDIM",

        "Codigo_Etiqueta",
        "Codigo_Etiqueta_FDIM",

        "_CAJAS_B1",
        "_CAJAS_B2",

        "CAMPOS_CAMBIADOS",
        "DETALLE_CAMBIOS"
    ]
].reset_index(drop=True)


print("=" * 85)
print("CONTROL 7A - DESTINATARIO / ETIQUETA")
print("=" * 85)

print("Casos encontrados:", len(df_control_7a))

display(df_control_7a)

print("\n✅ CONTROL 7A TERMINADO")
print("No se modificó ningún dato.")

CONTROL 7A - DESTINATARIO / ETIQUETA
Casos encontrados: 6


,_ORDEN_KEY,_ITEM_KEY,_FECHA_B1,FECHA_FDIM,Destinatario_Original,Destinatario_FDIM,Codigo_Etiqueta,Codigo_Etiqueta_FDIM,_CAJAS_B1,_CAJAS_B2,CAMPOS_CAMBIADOS,DETALLE_CAMBIOS
0,24290014,1,2026-08-21,2026-08-21,TRADER JOE'S LEBANON,TRADER JOE´S CALIFORNIA,Cafe,NARANJA,11.0,11.0,DESTINATARIO | CODIGO ETIQUETA,Destinatario: TRADER JOE'S LEBANON -> TRADER J...
1,24861451,1,2026-08-22,2026-08-22,WHOLE FOODS MARKET DISTRIBUTION CENTER OF AURO...,WHOLE FOODS MARKET DISTRIBUTION CENTER OF AURO...,Verde Obscuro,VERDE OBSCURO,28.0,28.0,DESTINATARIO,Destinatario: WHOLE FOODS MARKET DISTRIBUTION ...
2,24861504,1,2026-08-22,2026-08-22,(RDC) RICHMOND DISTRIBUTION CENTER WF,(RDC) RICHMOND DISTRIBUTION CENTER-RICHMOND. W...,Naranja,NARANJA,18.0,18.0,DESTINATARIO,Destinatario: (RDC) RICHMOND DISTRIBUTION CENT...
3,24868441,5,2026-08-22,2026-08-22,GIANT EAGLE CRAFTON PERISHABLE FACILITY,GIANT EAGLE CLEVELAND PERISHABLE FACILITY,Amarilla,AMARILLA,2.0,2.0,DESTINATARIO,Destinatario: GIANT EAGLE CRAFTON PERISHABLE F...
4,24872903,1,2026-08-20,2026-08-20,TRADER JOE'S MIAMI,TRADER JOE´S TEXAS,Rosada,ROJA,6.0,6.0,DESTINATARIO | CODIGO ETIQUETA,Destinatario: TRADER JOE'S MIAMI -> TRADER JO...
5,24872903,2,2026-08-20,2026-08-20,TRADER JOE'S MIAMI,TRADER JOE´S TEXAS,Rosada,ROJA,6.0,6.0,DESTINATARIO | CODIGO ETIQUETA,Destinatario: TRADER JOE'S MIAMI -> TRADER JO...



✅ CONTROL 7A TERMINADO
No se modificó ningún dato.


# BLOQUE 8

In [ ]:
# ============================================================
# BLOQUE 8
# PRECIO SISTER COMPANY
#
# REGLAS:
# - SOLO aplica a familia TRADER_JOES
# - NO modifica Precio VENTURE
# - Crea nueva variable: Precio SisterCompany
# - La tabla de precios es la fuente de verdad
# - GYP = GYPSOPHILA
# ============================================================


print("=" * 85)
print("BLOQUE 8 - PRECIO SISTER COMPANY")
print("=" * 85)


# ============================================================
# 1. LIMPIAR COLUMNAS DEL BLOQUE 8 SI SE VUELVE A EJECUTAR
# ============================================================

columnas_b8 = [
    "ITEM SisterCompany",
    "Precio SisterCompany",
    "Criterio SisterCompany",
    "Revisar SisterCompany"
]

df_venture_linea = df_venture_linea.drop(
    columns=[
        c for c in columnas_b8
        if c in df_venture_linea.columns
    ]
)


# ============================================================
# 2. PREPARAR TABLA MADRE DE PRECIOS
# ============================================================

df_precios_sister = df_precios_raw[
    ["ITEM", "TRADER"]
].copy()

df_precios_sister = df_precios_sister.dropna(
    subset=["ITEM"]
)

df_precios_sister["_ITEM_STD"] = (
    df_precios_sister["ITEM"]
    .apply(canon_texto)
)

df_precios_sister["_PRECIO"] = pd.to_numeric(
    df_precios_sister["TRADER"],
    errors="coerce"
)


# ============================================================
# 3. BLINDAJE EXTRA
# MISMO ITEM NO PUEDE TENER DOS PRECIOS DISTINTOS
# ============================================================

control_precios = (
    df_precios_sister
    .groupby("_ITEM_STD")["_PRECIO"]
    .nunique()
)

conflictos = control_precios[
    control_precios > 1
]

if len(conflictos) > 0:

    display(
        df_precios_sister[
            df_precios_sister["_ITEM_STD"].isin(
                conflictos.index
            )
        ]
    )

    raise ValueError(
        "❌ Hay ITEM Sister Company con precios diferentes."
    )


# ============================================================
# 4. DICCIONARIO DE PRECIOS
# ============================================================

precio_sister_dict = (
    df_precios_sister
    .drop_duplicates("_ITEM_STD")
    .set_index("_ITEM_STD")["_PRECIO"]
    .to_dict()
)


def buscar_precio_sister(item):

    if item is None or pd.isna(item):
        return pd.NA

    return precio_sister_dict.get(
        canon_texto(item),
        pd.NA
    )


# ============================================================
# 5. MAPEO DIRECTO PRODUCTO VENTURE -> ITEM SISTER
# ============================================================

MAPA_PRODUCTO_SISTER = {

    "ACHILLEA":
        "ACHILEA",

    "CRASPEDIAS":
        "CRASPEDIA",

    "DELPHINIUM":
        "DELPHINIUM",

    "DISBUD":
        "DISBUD : CREMON/SPIDER",

    "ERYNGIUM":
        "ERYNGIUM",

    "HYPERICUM":
        "HYPERICUM",

    "LEPIDIUM":
        "LEPIDIUM VIRGINICUM",

    "LIRIOSORIENTALES":
        "LIRIO ORIENTAL",

    "LISIANTHUS":
        "LISIANTHUS",

    "LYSIMACHIA":
        "LYSIMACHIA",

    "ORNITHOGALUM":
        "ORNITHOGALUM",

    "RANUNCULUS":
        "RANUNCULUS",

    "RICEFLOWER":
        "RICE FLOWER",

    "RUMEXCRISPUS":
        "RUMEX",

    "SOLIDAGO":
        "SOLIDAGO",

    "STATICE":
        "STATICE",

    "VERONICAS":
        "VERONICA",

    "VERONICASPRAY":
        "VERONICA SPRAY",

    # Estas tres ya sabemos que son HYBRIDS
    "SCABIOSABONBON":
        "SCABIOSA HYBRIDS",

    "SCABIOSAFOCAL":
        "SCABIOSA HYBRIDS",

    "SCABIOSAPINGPONG":
        "SCABIOSA HYBRIDS",
}


# ============================================================
# 6. RESOLVER PRECIO POR LÍNEA
# ============================================================

def resolver_precio_sister(row):

    # --------------------------------------------------------
    # NO ES TRADER JOE'S
    # --------------------------------------------------------

    if not row["APLICA_SISTER_COMPANY"]:

        return pd.Series({
            "ITEM SisterCompany": pd.NA,
            "Precio SisterCompany": pd.NA,
            "Criterio SisterCompany": "NO APLICA",
            "Revisar SisterCompany": False
        })


    producto = (
        row["PRODUCTO_STD"]
        if pd.notna(row["PRODUCTO_STD"])
        else ""
    )

    variedad = (
        row["VARIEDAD_STD"]
        if pd.notna(row["VARIEDAD_STD"])
        else ""
    )


    # ========================================================
    # A. PRODUCTOS DIRECTOS
    # ========================================================

    if producto in MAPA_PRODUCTO_SISTER:

        item = MAPA_PRODUCTO_SISTER[producto]
        precio = buscar_precio_sister(item)

        return pd.Series({
            "ITEM SisterCompany": item,
            "Precio SisterCompany": precio,
            "Criterio SisterCompany": "MAPEO DIRECTO",
            "Revisar SisterCompany": pd.isna(precio)
        })


    # ========================================================
    # B. GYPSOPHILA
    #
    # GYP = GYPSOPHILA
    # ========================================================

    if producto == "GYPSOPHILA":

        # Rainbow / Unicorn
        if (
            "RAINBOW" in variedad
            or "UNICORN" in variedad
        ):

            item = "GYP TINTED RAINBOW/UNICORN"


        # Xlence Tinted
        elif "TINTED" in variedad:

            item = "GYP TINTED"


        # Billion Lights:
        # todavía no tenemos regla comercial confirmada
        elif "BILLIONLIGHTS" in variedad:

            return pd.Series({
                "ITEM SisterCompany": pd.NA,
                "Precio SisterCompany": pd.NA,
                "Criterio SisterCompany":
                    "REVISAR: BILLION LIGHTS",
                "Revisar SisterCompany": True
            })


        # Gypsophila normal / Xlence
        else:

            item = "GYP"


        precio = buscar_precio_sister(item)

        return pd.Series({
            "ITEM SisterCompany": item,
            "Precio SisterCompany": precio,
            "Criterio SisterCompany":
                f"GYPSOPHILA - {row[COL_VARIEDAD1]}",
            "Revisar SisterCompany": pd.isna(precio)
        })


    # ========================================================
    # C. LIMONIUM
    # ========================================================

    if producto == "LIMONIUM":

        if (
            "OSHI" in variedad
            or "SKYLIGHT" in variedad
        ):

            item = "LIMONIUM OSHI Y EL SKYLIGHT"


        elif "TINTED" in variedad:

            item = "LIMONIUM TINTED"


        elif (
            "PINA" in variedad
            or "PINNA" in variedad
            or "COLADA" in variedad
        ):

            item = "LIMONIUM PIÑA COLADA"


        else:

            return pd.Series({
                "ITEM SisterCompany": pd.NA,
                "Precio SisterCompany": pd.NA,
                "Criterio SisterCompany":
                    f"REVISAR LIMONIUM: {row[COL_VARIEDAD1]}",
                "Revisar SisterCompany": True
            })


        precio = buscar_precio_sister(item)

        return pd.Series({
            "ITEM SisterCompany": item,
            "Precio SisterCompany": precio,
            "Criterio SisterCompany":
                f"LIMONIUM - {row[COL_VARIEDAD1]}",
            "Revisar SisterCompany": pd.isna(precio)
        })


    # ========================================================
    # D. SCABIOSA GENÉRICA
    # ========================================================

    if producto == "SCABIOSA":

        return pd.Series({
            "ITEM SisterCompany": pd.NA,
            "Precio SisterCompany": pd.NA,
            "Criterio SisterCompany":
                "REVISAR: CAUCASICA O HYBRIDS",
            "Revisar SisterCompany": True
        })


    # ========================================================
    # E. ROSES
    # ========================================================

    if producto == "ROSES":

        if (
            "PETITE" in variedad
            or "INTERMEDIATE" in variedad
        ):

            item = "PETITE ROSES E INTERMEDIATE"
            precio = buscar_precio_sister(item)

            return pd.Series({
                "ITEM SisterCompany": item,
                "Precio SisterCompany": precio,
                "Criterio SisterCompany":
                    "PETITE / INTERMEDIATE",
                "Revisar SisterCompany": pd.isna(precio)
            })


        return pd.Series({
            "ITEM SisterCompany": pd.NA,
            "Precio SisterCompany": pd.NA,
            "Criterio SisterCompany":
                f"REVISAR ROSES: {row[COL_VARIEDAD1]}",
            "Revisar SisterCompany": True
        })


    # ========================================================
    # F. PRODUCTO TRADER JOE'S SIN REGLA
    # ========================================================

    return pd.Series({
        "ITEM SisterCompany": pd.NA,
        "Precio SisterCompany": pd.NA,
        "Criterio SisterCompany":
            f"REVISAR PRODUCTO: {row[COL_PRODUCTO1]}",
        "Revisar SisterCompany": True
    })


# ============================================================
# 7. APLICAR
# ============================================================

resultado_sister = df_venture_linea.apply(
    resolver_precio_sister,
    axis=1
)

df_venture_linea = pd.concat(
    [
        df_venture_linea,
        resultado_sister
    ],
    axis=1
)


# ============================================================
# 8. BLINDAJE:
# PRECIO SISTER NUNCA PUEDE IR A OTRO CLIENTE
# ============================================================

precios_fuera_trader = (

    (~df_venture_linea["APLICA_SISTER_COMPANY"])
    &
    df_venture_linea["Precio SisterCompany"].notna()

).sum()


if precios_fuera_trader > 0:

    raise ValueError(
        "❌ Se asignó Precio SisterCompany "
        "a clientes que NO son Trader Joe's."
    )


# ============================================================
# 9. CONTROLES
# ============================================================

solo_trader = df_venture_linea[
    df_venture_linea["APLICA_SISTER_COMPANY"]
].copy()


print("\n" + "=" * 85)
print("RESUMEN SISTER COMPANY")
print("=" * 85)

print(
    "Líneas Trader Joe's:",
    len(solo_trader)
)

print(
    "Con Precio SisterCompany:",
    int(
        solo_trader[
            "Precio SisterCompany"
        ].notna().sum()
    )
)

print(
    "Requieren revisión:",
    int(
        solo_trader[
            "Revisar SisterCompany"
        ].sum()
    )
)

print(
    "Precio asignado fuera de Trader Joe's:",
    int(precios_fuera_trader)
)


# ============================================================
# 10. MUESTRA RESULTADO
# ============================================================

print("\n" + "=" * 85)
print("EJEMPLO PRECIO SISTER COMPANY")
print("=" * 85)

display(
    solo_trader[
        [
            COL_ORDEN1,
            COL_ITEM1,
            COL_DEST1,

            COL_PRODUCTO1,
            COL_VARIEDAD1,

            COL_PRECIO1,

            "ITEM SisterCompany",
            "Precio SisterCompany",

            "Criterio SisterCompany",
            "Revisar SisterCompany"
        ]
    ]
    .head(30)
    .reset_index(drop=True)
)


# ============================================================
# 11. SOLO LOS QUE NECESITAN REVISIÓN
# ============================================================

df_sister_revision = solo_trader[
    solo_trader["Revisar SisterCompany"]
].copy()


print("\n" + "=" * 85)
print("SISTER COMPANY - CASOS PARA REVISAR")
print("=" * 85)

print(
    "Total:",
    len(df_sister_revision)
)

display(
    df_sister_revision[
        [
            COL_ORDEN1,
            COL_ITEM1,
            COL_DEST1,
            COL_PRODUCTO1,
            COL_VARIEDAD1,

            "ITEM SisterCompany",
            "Precio SisterCompany",
            "Criterio SisterCompany"
        ]
    ]
    .reset_index(drop=True)
)


print("\n✅ BLOQUE 8 OK")

BLOQUE 8 - PRECIO SISTER COMPANY

RESUMEN SISTER COMPANY
Líneas Trader Joe's: 656
Con Precio SisterCompany: 656
Requieren revisión: 0
Precio asignado fuera de Trader Joe's: 0

EJEMPLO PRECIO SISTER COMPANY


,Orden\nCompra,No Item\n,Destinatario\n,Producto\n,Variedad\n,Precio\nUnitarios Sin Des,ITEM SisterCompany,Precio SisterCompany,Criterio SisterCompany,Revisar SisterCompany
0,24293427.0,1.0,TRADER JOE´S WASHINGTON,Gypsophila,Xlence,0.290,GYP,0.29,GYPSOPHILA - Xlence,False
1,24293436.0,1.0,TRADER JOE´S WASHINGTON,Gypsophila,Xlence Tinted,0.350,GYP TINTED,0.34,GYPSOPHILA - Xlence Tinted,False
2,24713429.0,9.0,TRADER JOE´S WASHINGTON,Roses,Roses Petite Hot Pink,0.155,PETITE ROSES E INTERMEDIATE,0.155,PETITE / INTERMEDIATE,False
3,24713429.0,9.0,TRADER JOE´S WASHINGTON,Roses,Roses Petite Yellow,0.155,PETITE ROSES E INTERMEDIATE,0.155,PETITE / INTERMEDIATE,False
4,24713429.0,9.0,TRADER JOE´S WASHINGTON,Roses,Roses Petite Orange,0.155,PETITE ROSES E INTERMEDIATE,0.155,PETITE / INTERMEDIATE,False
5,24713429.0,9.0,TRADER JOE´S WASHINGTON,Roses,Roses Petite Hot Pink,0.155,PETITE ROSES E INTERMEDIATE,0.155,PETITE / INTERMEDIATE,False
6,24713429.0,9.0,TRADER JOE´S WASHINGTON,Roses,Roses Petite Red,0.155,PETITE ROSES E INTERMEDIATE,0.155,PETITE / INTERMEDIATE,False
7,24713429.0,9.0,TRADER JOE´S WASHINGTON,Roses,Roses Petite Pink,0.155,PETITE ROSES E INTERMEDIATE,0.155,PETITE / INTERMEDIATE,False
8,24713429.0,9.0,TRADER JOE´S WASHINGTON,Roses,Roses Petite Pink,0.155,PETITE ROSES E INTERMEDIATE,0.155,PETITE / INTERMEDIATE,False
9,24713429.0,9.0,TRADER JOE´S WASHINGTON,Roses,Roses Petite Yellow,0.155,PETITE ROSES E INTERMEDIATE,0.155,PETITE / INTERMEDIATE,False



SISTER COMPANY - CASOS PARA REVISAR
Total: 0


,Orden\nCompra,No Item\n,Destinatario\n,Producto\n,Variedad\n,ITEM SisterCompany,Precio SisterCompany,Criterio SisterCompany



✅ BLOQUE 8 OK


# BLOQUE 9

In [ ]:
# ============================================================
# BLOQUE 9
# QA FINAL / CONTROL DE CALIDAD
#
# NO MODIFICA DATOS
# Verifica que todo esté listo para exportar.
# ============================================================

print("=" * 90)
print("BLOQUE 9 - QA FINAL")
print("=" * 90)


# ============================================================
# 1. FUNCIÓN PARA REGISTRAR CONTROLES
# ============================================================

qa = []

def registrar_control(control, valor, esperado, ok, tipo="CRITICO"):

    qa.append({
        "CONTROL": control,
        "VALOR": valor,
        "ESPERADO": esperado,
        "ESTADO": "OK" if ok else (
            "ADVERTENCIA" if tipo == "ADVERTENCIA" else "ERROR"
        )
    })


# ============================================================
# 2. DUPLICADOS VENTURE
#
# Orden + Item + Fecha debe ser único.
# ============================================================

dup_venture = int(
    df_venture_comp.duplicated(
        subset=[
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY"
        ]
    ).sum()
)

registrar_control(
    "Duplicados VENTURE Orden + Item + Fecha",
    dup_venture,
    0,
    dup_venture == 0
)


# ============================================================
# 3. DUPLICADOS FDIM
# ============================================================

dup_fdim_fecha = int(
    df_fdim_comp.duplicated(
        subset=[
            "_ORDEN_KEY",
            "_ITEM_KEY",
            "_FECHA_KEY"
        ]
    ).sum()
)

registrar_control(
    "Duplicados FDIM Orden + Item + Fecha",
    dup_fdim_fecha,
    0,
    dup_fdim_fecha == 0
)


# ============================================================
# 4. FDIM DEBE SER ÚNICO POR ORDEN + ITEM
#
# Esto es necesario porque el comparador final cruza
# VENTURE contra FDIM por Orden + Item.
# ============================================================

dup_fdim_order_item = int(
    df_fdim_comp.duplicated(
        subset=[
            "_ORDEN_KEY",
            "_ITEM_KEY"
        ]
    ).sum()
)

registrar_control(
    "Duplicados FDIM Orden + Item",
    dup_fdim_order_item,
    0,
    dup_fdim_order_item == 0
)


# ============================================================
# 5. ESTADOS DEL COMPARADOR
# ============================================================

estados_validos = {
    "SIN CAMBIO",
    "CAMBIO",
    "NO ENCONTRADO"
}

estados_actuales = set(
    df_comparacion["RESULTADO"]
    .dropna()
    .unique()
)

estados_invalidos = (
    estados_actuales
    -
    estados_validos
)

registrar_control(
    "Estados inválidos en comparación",
    ", ".join(estados_invalidos) if estados_invalidos else "Ninguno",
    "Ninguno",
    len(estados_invalidos) == 0
)


# ============================================================
# 6. TOTAL DE FILAS
# ============================================================

total_resultados = (
    df_comparacion["RESULTADO"]
    .value_counts()
    .sum()
)

registrar_control(
    "Filas clasificadas",
    int(total_resultados),
    len(df_comparacion),
    total_resultados == len(df_comparacion)
)


# ============================================================
# 7. CAMBIO DEBE TENER AL MENOS 1 CAMPO CAMBIADO
# ============================================================

cambios_sin_detalle = int(
    (
        (df_comparacion["RESULTADO"] == "CAMBIO")
        &
        (df_comparacion["NUM_CAMBIOS"] <= 0)
    ).sum()
)

registrar_control(
    "CAMBIO sin campo cambiado",
    cambios_sin_detalle,
    0,
    cambios_sin_detalle == 0
)


# ============================================================
# 8. SIN CAMBIO NO DEBE TENER CAMBIOS
# ============================================================

ok_con_cambios = int(
    (
        (df_comparacion["RESULTADO"] == "SIN CAMBIO")
        &
        (df_comparacion["NUM_CAMBIOS"] > 0)
    ).sum()
)

registrar_control(
    "SIN CAMBIO con diferencias registradas",
    ok_con_cambios,
    0,
    ok_con_cambios == 0
)


# ============================================================
# 9. SISTER COMPANY:
# NUNCA PUEDE ASIGNARSE A OTROS CLIENTES
# ============================================================

precio_fuera_trader = int(
    (
        (~df_venture_linea["APLICA_SISTER_COMPANY"])
        &
        df_venture_linea["Precio SisterCompany"].notna()
    ).sum()
)

registrar_control(
    "Precio SisterCompany fuera de Trader Joe's",
    precio_fuera_trader,
    0,
    precio_fuera_trader == 0
)


# ============================================================
# 10. ITEM SISTER TAMPOCO DEBE IR A OTROS CLIENTES
# ============================================================

item_fuera_trader = int(
    (
        (~df_venture_linea["APLICA_SISTER_COMPANY"])
        &
        df_venture_linea["ITEM SisterCompany"].notna()
    ).sum()
)

registrar_control(
    "ITEM SisterCompany fuera de Trader Joe's",
    item_fuera_trader,
    0,
    item_fuera_trader == 0
)


# ============================================================
# 11. TRADER JOE'S RESUELTOS SIN PRECIO
#
# Si Revisar=False, debería existir precio.
# ============================================================

trader_sin_precio_inexplicable = int(
    (
        df_venture_linea["APLICA_SISTER_COMPANY"]
        &
        (~df_venture_linea["Revisar SisterCompany"])
        &
        df_venture_linea["Precio SisterCompany"].isna()
    ).sum()
)

registrar_control(
    "Trader Joe's marcado OK pero sin Precio SisterCompany",
    trader_sin_precio_inexplicable,
    0,
    trader_sin_precio_inexplicable == 0
)


# ============================================================
# 12. CASOS SISTER QUE REQUIEREN REVISIÓN
#
# NO es error.
# Se reporta como advertencia.
# ============================================================

total_revision_sister = int(
    (
        df_venture_linea["APLICA_SISTER_COMPANY"]
        &
        df_venture_linea["Revisar SisterCompany"]
    ).sum()
)

registrar_control(
    "Sister Company pendientes de revisión",
    total_revision_sister,
    "Idealmente 0",
    total_revision_sister == 0,
    tipo="ADVERTENCIA"
)


# ============================================================
# 13. SIN COLOR FDIM
#
# Permitido por regla de negocio.
# Solo informativo.
# ============================================================

sin_color_fdim = int(
    (
        df_comparacion["OBSERVACION_FDIM"]
        .fillna("")
        .str.contains(
            "SIN COLOR FDIM",
            na=False
        )
    ).sum()
)

registrar_control(
    "Casos SIN COLOR FDIM",
    sin_color_fdim,
    "Permitido",
    True
)


# ============================================================
# 14. RESUMEN QA
# ============================================================

qa_resumen = pd.DataFrame(qa)

print("\n" + "=" * 90)
print("RESULTADO QA")
print("=" * 90)

display(qa_resumen)


# ============================================================
# 15. RESUMEN DEL COMPARADOR
# ============================================================

print("\n" + "=" * 90)
print("RESUMEN VENTURE VS FDIM")
print("=" * 90)

resumen_final_comparacion = (
    df_comparacion["RESULTADO"]
    .value_counts()
    .rename_axis("RESULTADO")
    .reset_index(name="CANTIDAD")
)

display(resumen_final_comparacion)


# ============================================================
# 16. RESUMEN SISTER COMPANY
# ============================================================

solo_trader_qa = df_venture_linea[
    df_venture_linea["APLICA_SISTER_COMPANY"]
].copy()

resumen_sister = pd.DataFrame({

    "INDICADOR": [
        "Líneas Trader Joe's",
        "Con Precio SisterCompany",
        "Pendientes de revisión",
        "Precio fuera de Trader Joe's"
    ],

    "CANTIDAD": [
        len(solo_trader_qa),

        int(
            solo_trader_qa[
                "Precio SisterCompany"
            ].notna().sum()
        ),

        total_revision_sister,

        precio_fuera_trader
    ]
})


print("\n" + "=" * 90)
print("RESUMEN SISTER COMPANY")
print("=" * 90)

display(resumen_sister)


# ============================================================
# 17. SI HAY ERRORES CRÍTICOS, DETENER
# ============================================================

errores_criticos = qa_resumen[
    qa_resumen["ESTADO"] == "ERROR"
]

if len(errores_criticos) > 0:

    print("\n❌ QA NO APROBADO")
    print("Hay controles críticos que debemos corregir.")

    display(errores_criticos)

    raise ValueError(
        "QA final falló. No exportar Excel todavía."
    )


# ============================================================
# 18. RESULTADO FINAL
# ============================================================

print("\n" + "=" * 90)

if total_revision_sister > 0:

    print("✅ QA TÉCNICO APROBADO")
    print(
        f"⚠️ Hay {total_revision_sister} líneas Trader Joe's "
        "que requieren revisión de precio."
    )

else:

    print("✅ QA COMPLETO APROBADO")
    print("✅ Todos los precios Sister Company fueron resueltos.")


print("✅ Sistema listo para BLOQUE 10 - EXPORTACIÓN EXCEL")
print("=" * 90)

BLOQUE 9 - QA FINAL

RESULTADO QA


,CONTROL,VALOR,ESPERADO,ESTADO
0,Duplicados VENTURE Orden + Item + Fecha,0,0,OK
1,Duplicados FDIM Orden + Item + Fecha,0,0,OK
2,Duplicados FDIM Orden + Item,0,0,OK
3,Estados inválidos en comparación,Ninguno,Ninguno,OK
4,Filas clasificadas,366,366,OK
5,CAMBIO sin campo cambiado,0,0,OK
6,SIN CAMBIO con diferencias registradas,0,0,OK
7,Precio SisterCompany fuera de Trader Joe's,0,0,OK
8,ITEM SisterCompany fuera de Trader Joe's,0,0,OK
9,Trader Joe's marcado OK pero sin Precio Sister...,0,0,OK



RESUMEN VENTURE VS FDIM


,RESULTADO,CANTIDAD
0,SIN CAMBIO,265
1,NO ENCONTRADO,61
2,CAMBIO,40



RESUMEN SISTER COMPANY


,INDICADOR,CANTIDAD
0,Líneas Trader Joe's,656
1,Con Precio SisterCompany,656
2,Pendientes de revisión,0
3,Precio fuera de Trader Joe's,0



✅ QA COMPLETO APROBADO
✅ Todos los precios Sister Company fueron resueltos.
✅ Sistema listo para BLOQUE 10 - EXPORTACIÓN EXCEL


# BLOQUE 10

In [ ]:
# ============================================================
# BLOQUE 10
# EXPORTACIÓN EXCEL FINAL
#
# NO modifica lógica.
# Solo organiza y exporta los resultados finales.
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files


print("=" * 90)
print("BLOQUE 10 - GENERANDO EXCEL FINAL")
print("=" * 90)


# ============================================================
# 1. FUNCIÓN SEGURA PARA TOMAR COLUMNAS
# ============================================================

def serie_segura(df, columna, valor_default=pd.NA):

    if columna in df.columns:
        return df[columna]

    return pd.Series(
        [valor_default] * len(df),
        index=df.index
    )


# ============================================================
# 2. REPORTE PRINCIPAL DE VALIDACIÓN
# ============================================================

reporte_validacion = pd.DataFrame({

    # --------------------------------------------------------
    # RESULTADO
    # --------------------------------------------------------

    "ESTADO":
        serie_segura(
            df_comparacion,
            "RESULTADO"
        ).replace({
            "SIN CAMBIO": "OK"
        }),

    "REVISION":
        serie_segura(
            df_comparacion,
            "DETALLE_CAMBIOS"
        ),

    "CAMPOS CAMBIADOS":
        serie_segura(
            df_comparacion,
            "CAMPOS_CAMBIADOS"
        ),

    "N CAMBIOS":
        serie_segura(
            df_comparacion,
            "NUM_CAMBIOS"
        ),


    # --------------------------------------------------------
    # LLAVE
    # --------------------------------------------------------

    "Orden Compra":
        serie_segura(
            df_comparacion,
            "_ORDEN_KEY"
        ),

    "No Item":
        serie_segura(
            df_comparacion,
            "_ITEM_KEY"
        ),


    # --------------------------------------------------------
    # FECHA
    # --------------------------------------------------------

    "Fecha VENTURE":
        serie_segura(
            df_comparacion,
            "_FECHA_B1"
        ),

    "Fecha FDIM":
        serie_segura(
            df_comparacion,
            "FECHA_FDIM"
        ),


    # --------------------------------------------------------
    # DESTINATARIO
    # NOMBRES ORIGINALES
    # --------------------------------------------------------

    "Destinatario VENTURE":
        serie_segura(
            df_comparacion,
            "Destinatario_Original"
        ),

    "Destinatario FDIM":
        serie_segura(
            df_comparacion,
            "Destinatario_FDIM"
        ),


    # --------------------------------------------------------
    # ETIQUETA
    # --------------------------------------------------------

    "Etiqueta VENTURE":
        serie_segura(
            df_comparacion,
            "Codigo_Etiqueta"
        ),

    "Etiqueta FDIM":
        serie_segura(
            df_comparacion,
            "Codigo_Etiqueta_FDIM"
        ),

    "Warehouse FDIM":
        serie_segura(
            df_comparacion,
            "Warehouse_FDIM"
        ),

    "Observación FDIM":
        serie_segura(
            df_comparacion,
            "OBSERVACION_FDIM"
        ),


    # --------------------------------------------------------
    # CAJAS
    # --------------------------------------------------------

    "Cajas VENTURE":
        serie_segura(
            df_comparacion,
            "_CAJAS_B1"
        ),

    "Cajas FDIM":
        serie_segura(
            df_comparacion,
            "_CAJAS_B2"
        ),


    # --------------------------------------------------------
    # INFORMACIÓN VENTURE
    # --------------------------------------------------------

    "Productos VENTURE":
        serie_segura(
            df_comparacion,
            "Productos"
        ),

    "Variedades VENTURE":
        serie_segura(
            df_comparacion,
            "Variedades"
        ),

    "N líneas producto":
        serie_segura(
            df_comparacion,
            "N_LINEAS_PRODUCTO"
        ),

    "Familia Cliente":
        serie_segura(
            df_comparacion,
            "FAMILIA_CLIENTE"
        ),

    "Aplica Sister Company":
        serie_segura(
            df_comparacion,
            "APLICA_SISTER_COMPANY"
        ),


    # --------------------------------------------------------
    # INFORMACIÓN FDIM
    # --------------------------------------------------------

    "Status FDIM":
        serie_segura(
            df_comparacion,
            "Status"
        ),

    "StatusDetail FDIM":
        serie_segura(
            df_comparacion,
            "StatusDetail"
        ),

    "SKU FDIM":
        serie_segura(
            df_comparacion,
            "SKU"
        ),

    "Description SKU FDIM":
        serie_segura(
            df_comparacion,
            "Description (SKU)"
        ),

})


# ============================================================
# 3. ORDENAR REPORTE
# ============================================================

prioridad_estado = {
    "CAMBIO": 1,
    "NO ENCONTRADO": 2,
    "OK": 3
}


reporte_validacion["_PRIORIDAD"] = (
    reporte_validacion["ESTADO"]
    .map(prioridad_estado)
)


reporte_validacion = (
    reporte_validacion
    .sort_values(
        [
            "_PRIORIDAD",
            "Orden Compra",
            "No Item",
            "Fecha VENTURE"
        ]
    )
    .drop(columns="_PRIORIDAD")
    .reset_index(drop=True)
)


# ============================================================
# 4. SUBREPORTES
# ============================================================

df_excel_cambios = (
    reporte_validacion[
        reporte_validacion["ESTADO"] == "CAMBIO"
    ]
    .copy()
)


df_excel_no_encontrados = (
    reporte_validacion[
        reporte_validacion["ESTADO"] == "NO ENCONTRADO"
    ]
    .copy()
)


df_excel_ok = (
    reporte_validacion[
        reporte_validacion["ESTADO"] == "OK"
    ]
    .copy()
)


df_excel_sin_color = (
    reporte_validacion[
        reporte_validacion[
            "Observación FDIM"
        ]
        .fillna("")
        .str.contains(
            "SIN COLOR FDIM",
            na=False
        )
    ]
    .copy()
)


# ============================================================
# 5. SISTER COMPANY
#
# SOLO TRADER JOE'S
# ============================================================

df_sister_excel = (
    df_venture_linea[
        df_venture_linea[
            "APLICA_SISTER_COMPANY"
        ]
    ]
    .copy()
)


df_sister_final = pd.DataFrame({

    "Orden Compra":
        df_sister_excel[
            "_ORDEN_KEY"
        ],

    "No Item":
        df_sister_excel[
            "_ITEM_KEY"
        ],

    "Fecha Salida":
        df_sister_excel[
            "_FECHA_KEY"
        ],

    "Destinatario":
        serie_segura(
            df_sister_excel,
            COL_DEST1 + "_EFECTIVO"
        ),

    "Familia Cliente":
        df_sister_excel[
            "FAMILIA_CLIENTE"
        ],

    "Producto":
        df_sister_excel[
            COL_PRODUCTO1
        ],

    "Variedad":
        df_sister_excel[
            COL_VARIEDAD1
        ],

    "Precio VENTURE":
        pd.to_numeric(
            df_sister_excel[
                COL_PRECIO1
            ],
            errors="coerce"
        ),

    "ITEM SisterCompany":
        df_sister_excel[
            "ITEM SisterCompany"
        ],

    "Precio SisterCompany":
        pd.to_numeric(
            df_sister_excel[
                "Precio SisterCompany"
            ],
            errors="coerce"
        ),

    "Criterio SisterCompany":
        df_sister_excel[
            "Criterio SisterCompany"
        ],

    "Requiere revisión":
        df_sister_excel[
            "Revisar SisterCompany"
        ]
})


df_sister_final = (
    df_sister_final
    .sort_values(
        [
            "Destinatario",
            "Orden Compra",
            "No Item",
            "Producto",
            "Variedad"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 6. TABLA MAESTRA DE PRECIOS UTILIZADA
# ============================================================

df_precios_excel = (
    df_precios_raw[
        [
            "ITEM",
            "TRADER"
        ]
    ]
    .copy()
)


df_precios_excel = df_precios_excel.rename(
    columns={
        "TRADER":
            "Precio SisterCompany"
    }
)


# ============================================================
# 7. RESUMEN PRINCIPAL
# ============================================================

total = len(reporte_validacion)

total_ok = len(df_excel_ok)

total_cambios = len(df_excel_cambios)

total_no = len(df_excel_no_encontrados)

total_sin_color = len(df_excel_sin_color)

total_trader = len(df_sister_final)

total_trader_precio = int(
    df_sister_final[
        "Precio SisterCompany"
    ]
    .notna()
    .sum()
)

total_sister_revision = int(
    df_sister_final[
        "Requiere revisión"
    ]
    .fillna(False)
    .sum()
)


# ============================================================
# 8. DESGLOSE DE CAMPOS CAMBIADOS
# ============================================================

cambios_explotados = (

    df_excel_cambios[
        "CAMPOS CAMBIADOS"
    ]
    .fillna("")
    .str.split(r"\s*\|\s*")
    .explode()
)


cambios_explotados = cambios_explotados[
    cambios_explotados.str.strip().ne("")
]


resumen_tipos_cambio = (
    cambios_explotados
    .value_counts()
    .rename_axis("CAMPO")
    .reset_index(name="CANTIDAD")
)


# ============================================================
# 9. CREAR EXCEL
# ============================================================

with pd.ExcelWriter(
    OUT_PATH,
    engine="xlsxwriter"
) as writer:

    workbook = writer.book


    # ========================================================
    # FORMATOS
    # ========================================================

    fmt_titulo = workbook.add_format({
        "bold": True,
        "font_size": 18,
        "font_color": "#FFFFFF",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter"
    })


    fmt_subtitulo = workbook.add_format({
        "bold": True,
        "font_size": 11,
        "font_color": "#44546A"
    })


    fmt_header_control = workbook.add_format({
        "bold": True,
        "font_color": "#FFFFFF",
        "bg_color": "#305496",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True
    })


    fmt_header_venture = workbook.add_format({
        "bold": True,
        "font_color": "#1F1F1F",
        "bg_color": "#D9EAF7",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True
    })


    fmt_header_fdim = workbook.add_format({
        "bold": True,
        "font_color": "#1F1F1F",
        "bg_color": "#DDEBF7",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True
    })


    fmt_header_sister = workbook.add_format({
        "bold": True,
        "font_color": "#1F1F1F",
        "bg_color": "#E2F0D9",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True
    })


    fmt_kpi_titulo = workbook.add_format({
        "bold": True,
        "font_color": "#FFFFFF",
        "bg_color": "#4472C4",
        "border": 1,
        "align": "center"
    })


    fmt_kpi_num = workbook.add_format({
        "bold": True,
        "font_size": 16,
        "border": 1,
        "align": "center"
    })


    fmt_ok = workbook.add_format({
        "bg_color": "#E2F0D9",
        "font_color": "#375623",
        "bold": True,
        "align": "center"
    })


    fmt_cambio = workbook.add_format({
        "bg_color": "#F4CCCC",
        "font_color": "#9C0006",
        "bold": True,
        "align": "center"
    })


    fmt_no = workbook.add_format({
        "bg_color": "#FCE4D6",
        "font_color": "#C65911",
        "bold": True,
        "align": "center"
    })


    fmt_observacion = workbook.add_format({
        "bg_color": "#FFF2CC",
        "font_color": "#7F6000"
    })


    fmt_precio = workbook.add_format({
        "num_format": "0.000"
    })


    fmt_cajas = workbook.add_format({
        "num_format": "0"
    })


    # ========================================================
    # FUNCIÓN DE ANCHOS
    # ========================================================

    def ajustar_anchos(sheet, df):

        for idx, col in enumerate(df.columns):

            max_len = len(str(col))

            if len(df) > 0:

                muestra = (
                    df[col]
                    .astype(str)
                    .replace("nan", "")
                    .head(500)
                )

                if len(muestra) > 0:
                    max_len = max(
                        max_len,
                        int(
                            muestra
                            .str.len()
                            .max()
                        )
                    )

            ancho = min(
                max(max_len + 2, 10),
                36
            )

            sheet.set_column(
                idx,
                idx,
                ancho
            )


    # ========================================================
    # FUNCIÓN PARA HOJAS DE DATOS
    # ========================================================

    def escribir_tabla(
        nombre,
        df,
        freeze_col=4,
        tiene_estado=True
    ):

        df.to_excel(
            writer,
            sheet_name=nombre,
            index=False,
            startrow=0
        )

        sheet = writer.sheets[nombre]

        sheet.freeze_panes(
            1,
            freeze_col
        )

        if len(df) > 0:

            sheet.autofilter(
                0,
                0,
                len(df),
                len(df.columns) - 1
            )


        # ----------------------------------------
        # ENCABEZADOS POR GRUPO
        # ----------------------------------------

        for col_num, columna in enumerate(df.columns):

            nombre_col = str(columna).upper()

            if (
                "VENTURE" in nombre_col
                or columna in [
                    "Orden Compra",
                    "No Item",
                    "Productos VENTURE",
                    "Variedades VENTURE"
                ]
            ):

                fmt = fmt_header_venture

            elif (
                "FDIM" in nombre_col
                or "WAREHOUSE" in nombre_col
            ):

                fmt = fmt_header_fdim

            elif (
                "SISTER" in nombre_col
            ):

                fmt = fmt_header_sister

            else:

                fmt = fmt_header_control


            sheet.write(
                0,
                col_num,
                columna,
                fmt
            )


        sheet.set_row(
            0,
            34
        )


        ajustar_anchos(
            sheet,
            df
        )


        # ----------------------------------------
        # ESTADO
        # ----------------------------------------

        if tiene_estado and "ESTADO" in df.columns:

            col_estado = df.columns.get_loc(
                "ESTADO"
            )

            sheet.conditional_format(
                1,
                col_estado,
                len(df),
                col_estado,
                {
                    "type": "text",
                    "criteria": "containing",
                    "value": "OK",
                    "format": fmt_ok
                }
            )

            sheet.conditional_format(
                1,
                col_estado,
                len(df),
                col_estado,
                {
                    "type": "text",
                    "criteria": "containing",
                    "value": "CAMBIO",
                    "format": fmt_cambio
                }
            )

            sheet.conditional_format(
                1,
                col_estado,
                len(df),
                col_estado,
                {
                    "type": "text",
                    "criteria": "containing",
                    "value": "NO ENCONTRADO",
                    "format": fmt_no
                }
            )


        # ----------------------------------------
        # OBSERVACIÓN SIN COLOR
        # ----------------------------------------

        if "Observación FDIM" in df.columns:

            c = df.columns.get_loc(
                "Observación FDIM"
            )

            sheet.conditional_format(
                1,
                c,
                len(df),
                c,
                {
                    "type": "text",
                    "criteria": "containing",
                    "value": "SIN COLOR FDIM",
                    "format": fmt_observacion
                }
            )


        # ----------------------------------------
        # FORMATOS NUMÉRICOS
        # ----------------------------------------

        for col in [
            "Cajas VENTURE",
            "Cajas FDIM"
        ]:

            if col in df.columns:

                c = df.columns.get_loc(col)

                sheet.set_column(
                    c,
                    c,
                    14,
                    fmt_cajas
                )


        for col in [
            "Precio VENTURE",
            "Precio SisterCompany"
        ]:

            if col in df.columns:

                c = df.columns.get_loc(col)

                sheet.set_column(
                    c,
                    c,
                    18,
                    fmt_precio
                )


    # ========================================================
    # 10. HOJA RESUMEN
    # ========================================================

    sheet = workbook.add_worksheet(
        "RESUMEN"
    )

    writer.sheets["RESUMEN"] = sheet


    sheet.merge_range(
        "A1:H2",
        "VALIDACIÓN ASISTENTES DE VENTAS",
        fmt_titulo
    )


    sheet.write(
        "A4",
        "VENTURE vs FDIM",
        fmt_subtitulo
    )


    # KPI títulos
    kpis = [
        ("TOTAL DESPACHOS", total),
        ("OK", total_ok),
        ("CAMBIOS", total_cambios),
        ("NO ENCONTRADOS", total_no)
    ]


    columnas_kpi = [
        "A",
        "C",
        "E",
        "G"
    ]


    for (titulo, valor), col in zip(
        kpis,
        columnas_kpi
    ):

        sheet.merge_range(
            f"{col}6:{chr(ord(col)+1)}6",
            titulo,
            fmt_kpi_titulo
        )

        sheet.merge_range(
            f"{col}7:{chr(ord(col)+1)}8",
            valor,
            fmt_kpi_num
        )


    # Sister Company
    sheet.write(
        "A11",
        "SISTER COMPANY - TRADER JOE'S",
        fmt_subtitulo
    )


    kpis_sister = [
        ("LÍNEAS TRADER JOE'S", total_trader),
        ("CON PRECIO SISTER", total_trader_precio),
        ("PENDIENTES", total_sister_revision),
        ("SIN COLOR FDIM", total_sin_color)
    ]


    for (titulo, valor), col in zip(
        kpis_sister,
        columnas_kpi
    ):

        sheet.merge_range(
            f"{col}13:{chr(ord(col)+1)}13",
            titulo,
            fmt_kpi_titulo
        )

        sheet.merge_range(
            f"{col}14:{chr(ord(col)+1)}15",
            valor,
            fmt_kpi_num
        )


    # Tipos de cambio
    sheet.write(
        "A18",
        "DESGLOSE DE CAMPOS CAMBIADOS",
        fmt_subtitulo
    )


    if len(resumen_tipos_cambio) > 0:

        resumen_tipos_cambio.to_excel(
            writer,
            sheet_name="RESUMEN",
            startrow=19,
            startcol=0,
            index=False
        )

        sheet.write(
            19,
            0,
            "CAMPO",
            fmt_header_control
        )

        sheet.write(
            19,
            1,
            "CANTIDAD",
            fmt_header_control
        )


    # Reglas principales
    reglas = [
        ["Regla", "Definición"],
        [
            "Llave VENTURE",
            "Orden Compra + Item + Fecha. Misma fecha: cantidades se suman."
        ],
        [
            "Cruce VENTURE vs FDIM",
            "Orden Compra + Item."
        ],
        [
            "Campos comparados",
            "Fecha, Destinatario, Código Etiqueta y Cantidad de Cajas."
        ],
        [
            "SIN COLOR FDIM",
            "No genera cambio de etiqueta. Se conserva como observación."
        ],
        [
            "Sister Company",
            "Aplica únicamente a la familia TRADER JOE'S."
        ],
        [
            "Precio SisterCompany",
            "Variable adicional. No reemplaza el Precio VENTURE."
        ],
    ]


    fila_reglas = 19 + max(
        len(resumen_tipos_cambio) + 4,
        10
    )


    sheet.write(
        fila_reglas,
        0,
        "REGLAS DE NEGOCIO",
        fmt_subtitulo
    )


    for r, fila in enumerate(
        reglas,
        start=fila_reglas + 1
    ):

        sheet.write_row(
            r,
            0,
            fila
        )


    sheet.set_column(
        "A:A",
        30
    )

    sheet.set_column(
        "B:B",
        55
    )

    sheet.set_column(
        "C:H",
        16
    )


    # ========================================================
    # 11. HOJAS COMPARADOR
    # ========================================================

    escribir_tabla(
        "VALIDACION",
        reporte_validacion,
        freeze_col=6
    )

    escribir_tabla(
        "CAMBIOS",
        df_excel_cambios,
        freeze_col=6
    )

    escribir_tabla(
        "NO_ENCONTRADOS",
        df_excel_no_encontrados,
        freeze_col=6
    )

    escribir_tabla(
        "MAPEO_OK",
        df_excel_ok,
        freeze_col=6
    )

    escribir_tabla(
        "SIN_COLOR_FDIM",
        df_excel_sin_color,
        freeze_col=6
    )


    # ========================================================
    # 12. SISTER COMPANY
    # ========================================================

    escribir_tabla(
        "SISTER_COMPANY",
        df_sister_final,
        freeze_col=5,
        tiene_estado=False
    )


    sheet_sister = writer.sheets[
        "SISTER_COMPANY"
    ]


    # Resaltar pendientes de revisión
    if (
        "Requiere revisión"
        in df_sister_final.columns
    ):

        col_rev = (
            df_sister_final.columns
            .get_loc(
                "Requiere revisión"
            )
        )

        sheet_sister.conditional_format(
            1,
            col_rev,
            len(df_sister_final),
            col_rev,
            {
                "type": "cell",
                "criteria": "==",
                "value": True,
                "format": fmt_observacion
            }
        )


    # ========================================================
    # 13. PRECIOS MAESTRA
    # ========================================================

    escribir_tabla(
        "PRECIOS_MAESTRA",
        df_precios_excel,
        freeze_col=0,
        tiene_estado=False
    )


# ============================================================
# 14. CONTROL POST-EXPORTACIÓN
# ============================================================

print("\n" + "=" * 90)
print("EXCEL GENERADO")
print("=" * 90)

print(
    "Archivo:",
    OUT_PATH
)

print(
    "VALIDACION:",
    len(reporte_validacion)
)

print(
    "CAMBIOS:",
    len(df_excel_cambios)
)

print(
    "NO ENCONTRADOS:",
    len(df_excel_no_encontrados)
)

print(
    "MAPEO OK:",
    len(df_excel_ok)
)

print(
    "SIN COLOR FDIM:",
    len(df_excel_sin_color)
)

print(
    "SISTER COMPANY:",
    len(df_sister_final)
)


print("\n✅ BLOQUE 10 OK")
print("✅ PROYECTO COMPLETADO")


# ============================================================
# 15. DESCARGAR
# ============================================================

files.download(
    str(OUT_PATH)
)

BLOQUE 10 - GENERANDO EXCEL FINAL

EXCEL GENERADO
Archivo: /content/VALIDACION_ASISTENTES_VENTAS.xlsx
VALIDACION: 366
CAMBIOS: 40
NO ENCONTRADOS: 61
MAPEO OK: 265
SIN COLOR FDIM: 6
SISTER COMPANY: 656

✅ BLOQUE 10 OK
✅ PROYECTO COMPLETADO


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>